<a href="https://colab.research.google.com/github/mj87246-ai/kfa-ai-agent/blob/main/KFA_%EB%B0%B8%EB%A5%98%EC%97%90%EC%9D%B4%EC%85%98_v8_%EC%9E%90%EB%8F%99_A4%EB%B3%B4%EA%B3%A0%EC%84%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KFA 밸류에이션 모듈 v8 — 병렬 개발 / 인계 계약 분리

**목적:** 공통 엔진과 산업·기업분석이 아직 완성되지 않아도 농심 종목코드로 독립 테스트하며, 최종 통합 때 **맨 위 테스트 입력 셀 하나만 제거·교체**해 동일한 밸류에이션 로직을 호출하도록 구성합니다.

**원천 우선순위:** 공통표로 받은 재무 수치 → 공식 DART 계정과 정합성 대조 → 부족한 재무·주식수·최신 공개 실적 DART 보완 → 최신 확인 가능한 종가 → 코드가 가치 계산 → AI는 가정과 해석 → 검증 및 HTML 저장.

**범위:** 국내 12월 결산 비금융 상장사, 보통주 기준. 실시간 거래가격이 아니라 마지막 확인 가능한 종가를 사용합니다. 데이터와 가정이 부족한 모형은 보류하며 숫자를 임의로 만들지 않습니다.


**v8 추가:** Gemini 일시 오류(429 일시 제한/500·502·503·504)에 대해 단계별 자동 재시도 및 예비 모델 순환. 인증·영구 할당량 제한은 즉시 중단. Colab에서 ■ 버튼을 누르면 중지 가능.


## 0. 다른 파트와의 인계 계약 — 가장 먼저 확인할 내용

**필수 입력은 종목코드 하나입니다.** 독립 실행에선 농심 `004370`을 사용합니다. 통합 시 공통 엔진이 데이터를 주면 아래 입력 계약대로 전달하면 됩니다.

| 전달자 | 입력값 | 필수 여부 | 밸류에이션에서 사용하는 방법 |
|---|---|---|---|
| 공통 엔진 / 통합 담당 | `stock_code` (6자리) | **필수** | 대상 기업 식별, DART·주가 수집 |
| 공통 엔진 | `common_records` (15열 공통 표 CSV 경로 **또는** `pandas.DataFrame`) | 선택 | 전달된 표의 수치·출처·공시일을 검증하고 DART와 대조 |
| 공통 엔진 | `column_map` (외부 열 이름 → 표준 열 이름) | **열 이름이 다를 때만 필수** | 형식 변환; 매핑 불가/중복/누락은 중단 |
| 산업·기업분석 | `company_name`, `company_type`, `industry`, `analysis_asof`, `analysis_brief` | 선택 | 전달 시 기업 정체성 확인 및 AI 해석의 전제로 사용 |
| 기업분석 | `peer_codes` (비교기업 2개 이상) | 선택 | 동종기업 배수 비교; 없으면 과거 자기배수는 **조건부 참고** |

**공통표 열 규격(현재 템플릿):** `종목코드, 종목명, 항목, 값, 단위, 기준, 기준일, 공시일, 출처, 출처 URL, 신뢰도, 수집 방법, 수집 시각, 상태, 비고`. 여기서 15는 **열 개수**이며 필수 재무지표 15개가 아닙니다. 예를 들어 공통 엔진 열 이름이 `ticker`라면 `column_map={"ticker":"종목코드"}`처럼 적어야 합니다. *중요:* 실제 통합 표의 규격이 확정되지 않았으므로 현재 데이터프레임에 맞는 매핑을 나중에 한 번 검토해야 합니다. 억지 자동 매핑은 하지 않습니다.

**최종 통합 시 지울 부분:** 바로 다음의 `🧪 독립 테스트 입력` 셀 **하나만** 제거하고, 같은 이름의 `VALUATION_HANDOFF` 딕셔너리를 상위 워크플로에서 넘기면 됩니다. 그다음 `🔌 인계 해석` 셀부터 실행합니다. 아래 계산·검증·보고서 셀은 수정할 필요가 없습니다.


In [1]:
#@title 🧪 독립 테스트 입력 — 통합할 때 이 셀만 삭제 { display-mode: "form" }
# 농심 테스트: 유일한 필수 수동 입력은 종목코드입니다.
# 실제 재무제표 숫자를 임의로 입력하지 않습니다. 나머지는 DART/주가에서 가져옵니다.
VALUATION_HANDOFF = {
    "stock_code": "004370",       # 농심 테스트
    "common_records": None,       # 공통 엔진이 완성되면 경로 또는 DataFrame
    "column_map": {},             # 실제 공통 엔진 열 이름이 다를 때만 지정
    "company_name": None,         # 앞 파트가 완성되기 전까지 미입력
    "company_type": None,
    "industry": None,
    "analysis_asof": None,
    "analysis_brief": None,
    "peer_codes": [],             # 비교기업 미확정: 허위 동종기업을 만들지 않음
}


In [2]:
#@title 🔌 인계 해석 — 통합할 때도 유지 (입력 형식 검증) { display-mode: "form" }
from datetime import datetime
from collections.abc import Mapping
import re

if not isinstance(globals().get("VALUATION_HANDOFF"), Mapping):
    raise ValueError("VALUATION_HANDOFF 딕셔너리가 필요합니다. 독립 테스트 셀 또는 공통 엔진을 연결하세요.")
H = dict(VALUATION_HANDOFF)
STOCK_CODE = str(H.get("stock_code") or "").strip()
if not re.fullmatch(r"[0-9]{6}", STOCK_CODE):
    raise ValueError(f"stock_code는 6자리 국내 종목코드여야 합니다: {STOCK_CODE!r}")
COMMON_INPUT = H.get("common_records")
COMMON_CSV_PATH = COMMON_INPUT if COMMON_INPUT is not None else ""  # v6 하위 호환용 이름; DataFrame도 허용
COMMON_COLUMN_MAP = dict(H.get("column_map") or {})
COMMON_ONLY = bool(H.get("common_only", False))
YEAR = datetime.now().year - 1  # 가장 최근 공개된 연차 후보; 실제 공시시점은 코드에서 다시 검증
UPSTREAM_COMPANY_NAME = H.get("company_name") or ""
UPSTREAM_COMPANY_TYPE = H.get("company_type") or ""
UPSTREAM_INDUSTRY = H.get("industry") or ""
UPSTREAM_ANALYSIS_ASOF = H.get("analysis_asof") or ""
UPSTREAM_SOURCE_NOTE = H.get("source_note") or ""
UPSTREAM_ANALYSIS_BRIEF = H.get("analysis_brief") or ""
PEER_CODES = [str(x).zfill(6) for x in (H.get("peer_codes") or [])]
PEER_NOTE = "동종기업 미제공 시 동일기업 과거 배수 중앙값을 조건부 참고치로 사용"
SEGMENTS = H.get("segments") or {}
VERIFIED_FINANCIAL_OVERRIDES = H.get("verified_financial_overrides") or {}
VERIFIED_TTM_OVERRIDES = H.get("verified_ttm_overrides") or {}
# 아래 값은 모델 운용 및 검증 설정값이지 다른 파트에 요청할 정보가 아닙니다.
DCF_YEARS = 5
EXTREME_VALUE_LOW, EXTREME_VALUE_HIGH = 0.2, 5.0
PEER_MULT_LOWER, PEER_MULT_UPPER = 0.5, 2.0
GEMINI_MODEL = "gemini-3.8-flash"
GEMINI_FALLBACK_MODEL = "gemini-3.5-flash-lite"
# 일시적 장애만 무제한 재시도 (별도 횟수 한도 없음). Ctrl+C 또는 Colab ■로 중지
GEMINI_RETRY_FOREVER = True
GEMINI_BACKOFF_INITIAL_SECONDS = 5
GEMINI_BACKOFF_MAX_SECONDS = 120
GEMINI_RETRY_STATUS_INTERVAL = 1   # 실패마다 경과·횟수 로그
# 선택: 0이면 전체 대기 시간 무제한, 그 외 초 단위 안전 상한
GEMINI_MAX_TOTAL_WAIT_SECONDS = 0
GEMINI_REQUEST_TIMEOUT_MS = 30000
MAX_PRICE_AGE_DAYS = 7
print("✅ 대상:", STOCK_CODE, "| 공통표:", "전달됨" if COMMON_INPUT is not None and not (isinstance(COMMON_INPUT,str) and not COMMON_INPUT.strip()) else "없음 → DART 독립 실행")


✅ 대상: 004370 | 공통표: 없음 → DART 독립 실행


In [3]:
#@title 📦 설치 { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "finance-datareader", "python-dotenv", "google-genai"],
                      capture_output=True, text=True)
print("✅ 설치 완료" if proc.returncode == 0 else "❌ 설치 실패: 런타임 → 런타임 다시 시작 → 모두 실행")

✅ 설치 완료


In [4]:
#@title 🧰 공통 도구 (고치지 말고 실행만. 코드가 궁금하면 이 셀을 더블클릭) { display-mode: "form" }
"""KFA AI agent study: shared collection toolkit (2nd session template).

Every source, whatever its collection level (1 API, 2 library, 3 file,
4 crawling, 5 manual), ends in the same record table via finalize().
Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 목차] 이 셀은 고치지 않고 실행만 해요. 아래는 어떤 함수가 어디에 쓰이는지예요.

  공통 출구 (모든 단계가 마지막에 거치는 곳)
    collect_table()     받아온 표 → 공통 표 15칸. 내 소스 함수의 마지막 줄은 항상 이것
    finalize()          꼬리표 붙이기·숫자/날짜 정리·형식 통과/확인 필요 판정 (collect_table이 불러 씀)
    long_from_table()   칸마다 항목이 하나씩인 넓은 표 → 숫자 하나 = 한 줄 (collect_table이 불러 씀)
    check_records()     공통 표 모양이 맞는지 검사하고 요약 출력
    save_records()      CSV 두 개로 저장 (전부 / 확인 필요만)
    run_step()          예시 하나를 실행하고 ✅ ⏭ ❌ 요약표에 한 줄 남김 (실패해도 멈추지 않음)
    step_summary()      요약표 보여주기

  1단계 공식 API      read_json_table() · collect_dart() · collect_ecos() · get_key()
  2단계 라이브러리     (도구 없음: 라이브러리가 준 표를 collect_table에 바로)
  3단계 파일          read_table()
  4단계 크롤링        read_html_table() · robots_allowed() · find_col()
  5단계 수동 입력      (pd.DataFrame으로 직접) · mount_drive() · read_source_cards() · pick_sheet_title() · collect_from_cards()

  정리 도구           to_number() 숫자 바꾸기 · to_date() 날짜 바꾸기 · is_blank() 빈칸 판단
  약속된 값           NOT_APPLICABLE = "해당 없음" · SAME_AS_DATE = 공시일을 기준일과 같게
  멈춤 신호           SkipStep = 준비 부족이라 건너뜀(⏭) · CollectError = 실패(❌), 이유를 쉬운 말로
"""

import io
import json
import math
import numbers
import os
import re
import urllib.robotparser
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

COLUMNS = [
    "종목코드", "종목명", "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각", "상태", "비고",
]
REQUIRED = [
    "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각",
]
NOT_APPLICABLE = "해당 없음"
NA_ALLOWED = {"종목코드", "종목명", "기준", "공시일"}
RELIABILITY = ("상", "중", "하")
LEVELS = {1: "공식 API", 2: "파이썬 라이브러리", 3: "파일 다운로드", 4: "크롤링", 5: "수동 입력"}
PASSED, REVIEW = "형식 통과", "확인 필요"
SAME_AS_DATE = "same_as_date"
CHECKED_MARKS = {"o", "y", "yes", "✅", "확인"}
NULL_TOKENS = {"", "-", ".", "\u2014", "n/a", "na", "nan", "none", "null"}
KST = timezone(timedelta(hours=9))
USER_AGENT = "KFA-AI-agent-study/0.2 (+https://github.com/phm0305/KFA-ai-agent-study)"


# 실패(❌). 메시지가 조원 화면에 그대로 나가서 키·전체 주소를 넣지 않아요
class CollectError(Exception):
    """Shown to members verbatim, so it must never contain keys or full URLs."""


# 준비가 안 돼서 건너뜀(⏭). 키 없음, 설정 꺼짐, 사이트가 막음 등
class SkipStep(Exception):
    pass


# 지금 한국 시각 (수집 시각 칸에 씀)
def now_kst():
    return datetime.now(KST).strftime("%Y-%m-%d %H:%M")


# 빈칸인지 판단 (None, 빈 글자, NaN 모두 빈칸)
def is_blank(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


# '1,234' '(500)' '△500' '12.5%' → 숫자. 못 바꾸면 (None, 이유)
def to_number(value):
    if is_blank(value):
        return None, None
    if isinstance(value, bool):
        return None, f"숫자로 못 바꿈: '{value}'"
    if isinstance(value, numbers.Integral):
        return int(value), None
    if isinstance(value, numbers.Real):
        return (float(value), None) if math.isfinite(value) else (None, None)
    text = str(value).strip()
    if text.lower() in NULL_TOKENS:
        return None, None
    cleaned = text.replace(",", "").replace(" ", "").rstrip("%")
    negative = False
    if cleaned.startswith("(") and cleaned.endswith(")"):
        negative, cleaned = True, cleaned[1:-1]
    elif cleaned.startswith("△"):
        negative, cleaned = True, cleaned[1:]
    if re.fullmatch(r"[+-]?\d+", cleaned):
        number = int(cleaned)
    else:
        try:
            number = float(cleaned)
        except ValueError:
            return None, f"숫자로 못 바꿈: '{text}'"
        if not math.isfinite(number):
            return None, f"숫자로 못 바꿈: '{text}'"
    return (-number if negative else number), None


_DAY = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*[-./월]\s*(\d{1,2})\s*일?\.?")
_COMPACT_DAY = re.compile(r"(\d{4})(\d{2})(\d{2})")
_MONTH = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*월?\.?")
_COMPACT_MONTH = re.compile(r"(\d{4})(\d{2})")
_QUARTER = re.compile(r"(\d{4})\s*-?\s*Q([1-4])", re.IGNORECASE)
_YEAR = re.compile(r"(\d{4})\s*년?")
_TIME_SUFFIX = re.compile(r"[ T]\d{1,2}:\d{2}(:\d{2})?(\.\d+)?$")


# '2026.6.30.' '20260630' '2026년 6월' '2026Q2' 등 → 표준 날짜 글자. 못 바꾸면 (None, 이유)
def to_date(value):
    """Normalize to YYYY-MM-DD, YYYY-MM, YYYYQn or YYYY; anything else is a problem."""
    if is_blank(value):
        return None, None
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.strftime("%Y-%m-%d"), None
    text = _TIME_SUFFIX.sub("", str(value).strip())
    for pattern in (_DAY, _COMPACT_DAY):
        match = pattern.fullmatch(text)
        if match:
            try:
                return date(*map(int, match.groups())).isoformat(), None
            except ValueError:
                return None, f"없는 날짜: '{value}'"
    for pattern in (_MONTH, _COMPACT_MONTH):
        match = pattern.fullmatch(text)
        if match:
            year, month = map(int, match.groups())
            if 1 <= month <= 12:
                return f"{year:04d}-{month:02d}", None
            return None, f"없는 날짜: '{value}'"
    match = _QUARTER.fullmatch(text)
    if match:
        return f"{match.group(1)}Q{match.group(2)}", None
    match = _YEAR.fullmatch(text)
    if match:
        return match.group(1), None
    return None, f"날짜 모양을 모름: '{value}'"


# 이름에 그 글자가 들어간 첫 칸 찾기 (크롤링 표처럼 칸 이름이 조금씩 바뀔 때)
def find_col(table, keyword):
    for col in table.columns:
        if keyword in str(col):
            return col
    raise CollectError(f"'{keyword}'가 들어간 칸이 없어요. 지금 표의 칸: {[str(c) for c in table.columns]}")


# 넓은 표(칸마다 항목) → 긴 표(숫자 하나 = 한 줄). items·date_col을 여기서 씀
def long_from_table(table, items, date_col=None, keep=None):
    """Wide table (one column per item) -> one row per number.

    items: {raw column: (항목 이름, 단위)}. keep: {raw column: our column} copied per row.
    """
    if date_col is None:
        if isinstance(table.index, pd.RangeIndex):
            raise CollectError("날짜가 들어 있는 칸 이름을 date_col로 알려 주세요")
        table = table.reset_index()
        date_col = table.columns[0]
    keep = keep or {}
    missing = [c for c in [*items, date_col, *keep] if c not in table.columns]
    if missing:
        raise CollectError(f"표에 없는 칸: {missing}. 지금 표의 칸: {[str(c) for c in table.columns]}")
    parts = []
    for raw_col, (label, unit) in items.items():
        part = pd.DataFrame({
            "항목": label,
            "값": pd.Series(table[raw_col].tolist(), dtype=object),
            "단위": unit,
            "기준일": pd.Series(table[date_col].tolist(), dtype=object),
        })
        for raw, ours in keep.items():
            part[ours] = table[raw].tolist()
        parts.append(part)
    if not parts:
        raise CollectError("items가 비어 있어요. 어떤 칸을 숫자로 가져올지 적어 주세요")
    return pd.concat(parts, ignore_index=True)


# 꼬리표 채우기 + 숫자·날짜 정리 + 형식 통과/확인 필요 판정 (모든 단계 공통)
def finalize(table, *, level, source=None, source_url=None, reliability=None, basis=None,
             disclosed=None, base_date=None, stock_code=None, stock_name=None, unit=None, note=None):
    """Turn any table with at least 항목/값 into the shared record table.

    A cell already in the table wins over the keyword default. disclosed may be a
    date, NOT_APPLICABLE, or SAME_AS_DATE (copy 기준일, for prices and rates).
    """
    if level not in LEVELS:
        raise ValueError(f"level must be one of {list(LEVELS)}")
    defaults = {
        "종목코드": stock_code, "종목명": stock_name, "단위": unit, "기준": basis,
        "기준일": base_date, "출처": source, "출처 URL": source_url, "신뢰도": reliability,
    }
    collected_at = now_kst()
    method = f"{level} {LEVELS[level]}"
    out = []
    for raw in table.to_dict("records"):
        row = {col: raw.get(col) for col in COLUMNS}
        for col, default in defaults.items():
            if is_blank(row[col]) and default is not None:
                row[col] = default
        notes = [n for n in (raw.get("비고"), note) if not is_blank(n)]
        reasons = []

        if is_blank(row["공시일"]):
            if disclosed == SAME_AS_DATE:
                row["공시일"] = row["기준일"]
                notes.append("공시일은 기준일과 같다고 봄")
            elif disclosed is not None:
                row["공시일"] = disclosed
        row["수집 방법"] = method
        if is_blank(row["수집 시각"]):
            row["수집 시각"] = collected_at

        value, value_problem = to_number(row["값"])
        if value_problem:
            reasons.append(value_problem)
        else:
            row["값"] = value

        for col in ("기준일", "공시일"):
            if is_blank(row[col]) or row[col] == NOT_APPLICABLE:
                continue
            normalized, problem = to_date(row[col])
            if problem:
                reasons.append(f"{col} {problem}")
            else:
                row[col] = normalized

        for col in REQUIRED:
            if is_blank(row[col]) and not (col == "값" and value_problem):
                reasons.append(f"{col} 비어 있음")
        for col in COLUMNS:
            if row[col] == NOT_APPLICABLE and col not in NA_ALLOWED:
                reasons.append(f"{col}에는 '{NOT_APPLICABLE}'을 쓸 수 없음")

        base, disc = str(row["기준일"] or ""), str(row["공시일"] or "")
        if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", base) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", disc):
            if disc[:len(base)] < base:
                reasons.append("공시일이 기준일보다 빠름")
        if not is_blank(row["신뢰도"]) and row["신뢰도"] not in RELIABILITY:
            reasons.append("신뢰도는 상/중/하 중 하나")

        checked = str(raw.get("확인") or "").strip().lower() in CHECKED_MARKS
        if level == 4 and not checked:
            reasons.append("크롤링 값: 원문 화면과 한 번 대조")
        if level == 5 and not checked:
            reasons.append("손으로 옮긴 값: 원문과 대조")

        row["상태"] = REVIEW if reasons else PASSED
        row["비고"] = "; ".join([*dict.fromkeys(notes), *(f"⚠ {r}" for r in reasons)])
        out.append(row)

    records = pd.DataFrame(out, columns=COLUMNS)
    records["값"] = pd.Series([r["값"] for r in out], dtype=object)
    return records


# 모든 단계의 공통 출구: 받아온 표 → 공통 표 15칸
def collect_table(table, items=None, date_col=None, keep=None, **meta):
    """Single exit for every level: raw table in, record table out.

    Without items, the table must already have 항목/값 columns (typical for manual CSVs).
    """
    if not isinstance(table, pd.DataFrame):
        raise CollectError(f"표(DataFrame)가 아니라 {type(table).__name__}가 들어왔어요")
    if items is not None:
        table = long_from_table(table, items, date_col, keep)
    elif not {"항목", "값"} <= set(table.columns):
        raise CollectError(f"items 없이 쓰려면 '항목'과 '값' 칸이 있어야 해요. 지금 칸: {[str(c) for c in table.columns]}")
    return finalize(table, **meta)


# 확인 필요 줄의 ⚠ 이유를 종류별로 세기
def review_reasons(records):
    reasons = Counter()
    for note in records.loc[records["상태"] == REVIEW, "비고"].fillna(""):
        for part in note.split("; "):
            if part.startswith("⚠ "):
                reasons[re.sub(r"'.*'", "'…'", part[2:])] += 1
    return reasons


# 공통 표 모양 검사. 맞으면 '✅ 칸 모양 OK' 요약을 출력
def check_records(records, show=True):
    problems = []
    if not isinstance(records, pd.DataFrame):
        problems.append(f"표(DataFrame)가 아니에요: {type(records).__name__}")
    elif list(records.columns) != COLUMNS:
        missing = [c for c in COLUMNS if c not in records.columns]
        extra = [c for c in records.columns if c not in COLUMNS]
        problems.append(f"칸 이름·순서가 달라요 (빠진 칸 {missing}, 남는 칸 {extra})")
    else:
        bad_state = ~records["상태"].isin([PASSED, REVIEW])
        if bad_state.any():
            problems.append(f"상태 칸에 '{PASSED}'/'{REVIEW}' 말고 다른 값이 {int(bad_state.sum())}줄")
        blank = records[REQUIRED].apply(lambda col: col.map(is_blank)).any(axis=1)
        hidden = blank & (records["상태"] == PASSED)
        if hidden.any():
            problems.append(f"필수 칸이 비었는데 '{PASSED}'인 줄이 {int(hidden.sum())}개 (finalize를 거치지 않은 듯)")
    if show:
        if problems:
            for p in problems:
                print("❌", p)
        else:
            passed = int((records["상태"] == PASSED).sum())
            print(f"✅ 칸 모양 OK: {len(records)}줄 (형식 통과 {passed} · 확인 필요 {len(records) - passed})")
            for reason, count in review_reasons(records).most_common(5):
                print(f"   ⚠ {reason}: {count}줄")
    return not problems


# CSV 저장: 전부(stem.csv) + 확인 필요만(stem_확인필요.csv). 엑셀에서 안 깨지게 utf-8-sig
def save_records(records, stem="records"):
    all_path, review_path = f"{stem}.csv", f"{stem}_확인필요.csv"
    records.to_csv(all_path, index=False, encoding="utf-8-sig")
    records[records["상태"] == REVIEW].to_csv(review_path, index=False, encoding="utf-8-sig")
    return all_path, review_path


STEP_RESULTS = []


# 예시 하나 실행 → 요약표에 ✅ ⏭ ❌ 한 줄. 어떤 오류가 나도 '모두 실행'을 멈추지 않음
def run_step(label, level, func):
    """Run one source without ever stopping 'Run all'; errors become a summary row."""
    entry = {"단계": f"{level} {LEVELS[level]}", "예시": label, "결과": "❌", "줄 수": 0, "확인 필요": 0, "메모": ""}
    STEP_RESULTS[:] = [e for e in STEP_RESULTS if e["예시"] != label]
    STEP_RESULTS.append(entry)
    try:
        records = func()
    except SkipStep as e:
        entry.update(결과="⏭", 메모=str(e))
        print(f"⏭ {label}: {e}")
        return None
    except CollectError as e:
        entry["메모"] = str(e)
        print(f"❌ {label}: {e}")
        return None
    except requests.RequestException as e:
        entry["메모"] = f"인터넷 연결 문제로 못 받아왔어요 ({type(e).__name__}). 잠시 뒤 이 셀만 다시 실행해 주세요"
        print(f"❌ {label}: {entry['메모']}")
        return None
    except Exception as e:
        entry["메모"] = f"예상 못 한 오류 ({type(e).__name__}). 셀을 다시 실행해 보고, 그래도 안 되면 조 톡방에 공유"
        print(f"❌ {label}: {entry['메모']}")
        return None
    if not check_records(records, show=False):
        entry["메모"] = "공통 표 모양이 아니에요. check_records(결과)로 무엇이 다른지 확인"
        check_records(records)
        return None
    review = int((records["상태"] == REVIEW).sum())
    entry.update(결과="✅", **{"줄 수": len(records), "확인 필요": review})
    print(f"✅ {label}: {len(records)}줄 (확인 필요 {review}줄)")
    return records


# 지금까지 실행한 예시의 요약표
def step_summary():
    return pd.DataFrame(STEP_RESULTS, columns=["단계", "예시", "결과", "줄 수", "확인 필요", "메모"])


# 주소에 요청 보내기. 오류 메시지에는 사이트 이름만 넣어 키가 화면에 안 나오게 함
def http_get(url, params=None, timeout=30):
    host = urlparse(url).netloc
    try:
        resp = requests.get(url, params=params, timeout=timeout, headers={"User-Agent": USER_AGENT})
    except requests.Timeout:
        raise CollectError(f"{host} 응답이 {timeout}초 넘게 없어요. 잠시 뒤 다시 실행해 주세요") from None
    except requests.RequestException:
        raise CollectError(f"{host}에 연결하지 못했어요. 인터넷이나 사이트 상태를 확인해 주세요") from None
    if resp.status_code != 200:
        hint = {401: "키나 로그인이 필요해요", 403: "사이트가 접근을 막았어요", 404: "주소가 바뀌었을 수 있어요",
                429: "요청이 너무 잦아요. 1분 뒤 다시"}.get(resp.status_code, "")
        raise CollectError(f"{host}가 {resp.status_code} 응답을 줬어요. {hint}".strip())
    return resp


# [1단계] JSON 응답에서 'a.b.c' 위치의 목록을 표로. 위치가 틀리면 그 자리에 뭐가 있는지 알려줌
def read_json_table(url, rows_path, params=None):
    """Fetch JSON and return the list found at a dotted path (e.g. 'StatisticSearch.row') as a table."""
    try:
        data = http_get(url, params).json()
    except ValueError:
        raise CollectError("응답이 JSON이 아니에요. 주소의 형식(json/xml) 부분을 확인해 주세요") from None
    current = data
    for key in rows_path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
            continue
        here = list(current) if isinstance(current, dict) else type(current).__name__
        preview = json.dumps(data, ensure_ascii=False)[:200]
        raise CollectError(f"응답에서 '{key}'를 못 찾았어요. 그 자리에 있는 것: {here} / 응답 앞부분: {preview}")
    rows = [current] if isinstance(current, dict) else current
    if not isinstance(rows, list) or not rows:
        raise CollectError(f"'{rows_path}' 아래가 비어 있어요")
    return pd.json_normalize(rows)


DART_URL = "https://opendart.fss.or.kr/api"
DART_REPORT = {1: "11013", 2: "11012", 3: "11014", 4: "11011"}
QUARTER_END = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
DART_ERRORS = {
    "010": "등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요",
    "011": "사용할 수 없는 키예요",
    "013": "조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요",
    "020": "오늘 요청 한도를 넘었어요. 내일 다시 해 주세요",
    "100": "요청 값이 잘못됐어요 (연도·분기·연결/별도)",
    "800": "DART 시스템 점검 중이에요",
}
DART_ITEMS = {
    "ifrs-full_Revenue": ("매출액", True),
    "dart_OperatingIncomeLoss": ("영업이익", True),
    "ifrs-full_ProfitLoss": ("당기순이익", True),
    "ifrs-full_Assets": ("자산총계", False),
    "ifrs-full_Liabilities": ("부채총계", False),
    "ifrs-full_Equity": ("자본총계", False),
}
REPO_RAW_URL = "https://raw.githubusercontent.com/phm0305/KFA-ai-agent-study/main/"
DART_CORPS_FILE = "backup/dart_corps.csv"
_DART_CORPS = {}
_DART_CORPS_LOADED = set()


def _dart_status(data):
    status = data.get("status")
    if status != "000":
        raise CollectError(f"DART {status}: {DART_ERRORS.get(status, data.get('message', '알 수 없는 응답'))}")


def download_dart_corps(api_key):
    """Listed companies from DART's full corpCode.xml: {stock_code: (corp_code, corp_name)}."""
    resp = http_get(f"{DART_URL}/corpCode.xml", {"crtfc_key": api_key}, timeout=600)
    try:
        xml_bytes = zipfile.ZipFile(io.BytesIO(resp.content)).read("CORPCODE.xml")
    except zipfile.BadZipFile:
        try:
            _dart_status(resp.json())
        except ValueError:
            pass
        raise CollectError("DART 회사 번호표를 받지 못했어요. 키를 확인해 주세요") from None
    corps = {}
    for item in ET.fromstring(xml_bytes).iter("list"):
        code = (item.findtext("stock_code") or "").strip()
        if code:
            corps[code] = (item.findtext("corp_code"), item.findtext("corp_name"))
    return corps


def read_dart_corps(content):
    # dtype=str keeps leading zeros: read as numbers, "000660" would become 660 and never match.
    table = pd.read_csv(io.BytesIO(content), dtype=str, keep_default_na=False)
    return {row.stock_code: (row.corp_code, row.corp_name) for row in table.itertuples(index=False)}


# [1단계] 종목코드 → DART 회사 번호. 레포의 작은 상장사 표를 먼저 보고, 없을 때만 DART 전체 목록을 받음
def dart_corp(stock_code, api_key):
    if "repo" not in _DART_CORPS_LOADED:
        _DART_CORPS_LOADED.add("repo")
        try:
            _DART_CORPS.update(read_dart_corps(http_get(REPO_RAW_URL + DART_CORPS_FILE, timeout=10).content))
        except CollectError:
            pass
    if stock_code not in _DART_CORPS and "dart" not in _DART_CORPS_LOADED:
        print("DART 회사 목록을 받는 중이에요. 처음 한 번, 몇 분 걸릴 수 있어요")
        _DART_CORPS.update(download_dart_corps(api_key))
        _DART_CORPS_LOADED.add("dart")
    if stock_code not in _DART_CORPS:
        raise CollectError(f"{stock_code}: 상장사 목록에 없어요. 종목코드 6자리를 확인해 주세요")
    return _DART_CORPS[stock_code]


# [1단계] DART 재무제표 응답에서 6개 계정만 골라 3개월·누적을 나눠 적기
def dart_rows_to_long(rows, year, quarter):
    """Pick six accounts by standard account_id; flows get 3-month and cumulative rows."""
    base_note = "기준일은 12월 결산 가정"
    out, seen = [], set()
    for row in rows:
        account = row.get("account_id")
        if account not in DART_ITEMS or account in seen:
            continue
        name, is_flow = DART_ITEMS[account]
        if is_flow and row.get("sj_div") not in ("IS", "CIS"):
            continue
        seen.add(account)
        rcept_no = row.get("rcept_no", "")
        common = {
            "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
            "공시일": rcept_no[:8], "출처 URL": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        }
        if not is_flow:
            parts = [(name, row.get("thstrm_amount"), base_note)]
        elif quarter == 4:
            parts = [(f"{name}(연간)", row.get("thstrm_amount"), base_note + "; 4분기 3개월치 = 연간 - 3분기 누적")]
        elif quarter == 1:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note + "; 1분기는 3개월 = 누적")]
        else:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note),
                     (f"{name}(누적)", row.get("thstrm_add_amount"), base_note)]
        for label, amount, memo in parts:
            out.append({**common, "항목": label, "값": amount, "비고": memo})
    for account, (name, _) in DART_ITEMS.items():
        if account not in seen:
            out.append({"항목": name, "값": None, "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
                        "비고": f"이 회사 공시에서 표준 계정({account})을 못 찾음"})
    return pd.DataFrame(out)


# [1단계] DART 재무제표 → 공통 표 (매출액·영업이익·당기순이익·자산·부채·자본)
def collect_dart(stock_code, year, quarter, fs_div, api_key):
    if not api_key:
        raise SkipStep("DART 키가 없어요 (Secrets에 DART_API_KEY 등록 후 다시)")
    if quarter not in DART_REPORT or fs_div not in ("CFS", "OFS"):
        raise CollectError("QUARTER는 1~4, FS_DIV는 'CFS'(연결) 또는 'OFS'(별도)")
    corp_code, corp_name = dart_corp(stock_code, api_key)
    data = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
        "reprt_code": DART_REPORT[quarter], "fs_div": fs_div,
    }).json()
    _dart_status(data)
    return collect_table(
        dart_rows_to_long(data["list"], year, quarter),
        level=1, source="DART 단일회사 전체 재무제표 API", reliability="상",
        basis="연결" if fs_div == "CFS" else "별도", stock_code=stock_code, stock_name=corp_name,
        source_url="https://opendart.fss.or.kr",
    )


# [1단계] 한국은행 ECOS 통계 1개 → 공통 표. 키가 없으면 ⏭
def collect_ecos(api_key, stat_code, cycle, start, end, item_code, label, unit):
    if not api_key:
        raise SkipStep("ECOS 키가 없어요 (선택 사항: ecos.bok.or.kr에서 무료 발급 후 Secrets에 ECOS_API_KEY)")
    url = f"https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr/1/1000/{stat_code}/{cycle}/{start}/{end}/{item_code}"
    try:
        data = http_get(url).json()
    except ValueError:
        raise CollectError("ECOS 응답이 JSON이 아니에요") from None
    if "RESULT" in data:
        raise CollectError(f"ECOS: {data['RESULT'].get('MESSAGE', data['RESULT'])}")
    table = pd.json_normalize(data["StatisticSearch"]["row"])
    return collect_table(
        table, {"DATA_VALUE": (label, unit)}, date_col="TIME",
        level=1, source="한국은행 ECOS Open API", reliability="상", basis=NOT_APPLICABLE,
        source_url=f"https://ecos.bok.or.kr/#/SearchStat?statCode={stat_code}",
    )


# [3·5단계] 주소나 파일 경로의 CSV·엑셀 읽기 (한글 깨짐 cp949, 이름만 .xls인 파일도 처리)
# 칸 이름과 글자 값의 앞뒤 공백은 지워요 (화면에서 안 보이는 차이라서)
def read_table(source, sheet_name=0, skiprows=None):
    """CSV/Excel from a URL or a path. Handles cp949 CSVs and HTML disguised as .xls.

    Leading/trailing spaces in column names and text cells are removed: they are
    invisible on screen, so no reader can mean anything by them.
    """
    table = _read_table_raw(source, sheet_name, skiprows)
    table.columns = [c.strip() if isinstance(c, str) else c for c in table.columns]
    for col in table.columns:
        if not pd.api.types.is_numeric_dtype(table[col]):
            table[col] = table[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    return table


def _read_table_raw(source, sheet_name, skiprows):
    if str(source).startswith(("http://", "https://")):
        content, name = http_get(str(source)).content, urlparse(str(source)).path.lower()
    else:
        path = Path(source)
        if not path.exists():
            raise CollectError(f"파일이 없어요: {path.name}. 경로와 파일 이름(띄어쓰기·확장자)을 확인해 주세요")
        content, name = path.read_bytes(), path.name.lower()
    if content.lstrip()[:1] == b"<":
        try:
            return pd.read_html(io.StringIO(content.decode("utf-8", errors="replace")))[0]
        except ValueError:
            raise CollectError("파일 안이 표가 아닌 웹페이지예요. 받은 파일을 열어 확인해 주세요") from None
    if name.endswith((".xlsx", ".xlsm", ".xls")):
        try:
            return pd.read_excel(io.BytesIO(content), sheet_name=sheet_name, skiprows=skiprows)
        except ImportError:
            raise CollectError("이 엑셀 형식을 읽는 도구가 없어요. 엑셀에서 .xlsx로 다시 저장해 주세요") from None
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(io.BytesIO(content), encoding=encoding, skiprows=skiprows)
        except UnicodeDecodeError:
            continue
    raise CollectError("CSV 글자를 못 읽었어요 (utf-8, cp949 둘 다 실패). 엑셀에서 'CSV UTF-8'로 다시 저장해 주세요")


# [4단계] 사이트의 robots.txt가 코드 수집을 허락하는지 (모르면 None)
def robots_allowed(url):
    """True/False from robots.txt; None when it cannot be read (caller decides)."""
    parts = urlparse(url)
    try:
        resp = requests.get(f"{parts.scheme}://{parts.netloc}/robots.txt", timeout=15,
                            headers={"User-Agent": USER_AGENT})
    except requests.RequestException:
        return None
    if resp.status_code in (401, 403):
        return False
    if resp.status_code >= 400:
        return True
    parser = urllib.robotparser.RobotFileParser()
    parser.parse(resp.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


# [4단계] 웹페이지 화면의 표 읽기. robots.txt가 막으면 ⏭
def read_html_table(url, match=".+", index=0):
    host = urlparse(url).netloc
    allowed = robots_allowed(url)
    if allowed is False:
        raise SkipStep(f"{host}의 robots.txt가 코드 수집을 막고 있어요. 이 자료는 5단계(수동)로 옮겨 주세요")
    if allowed is None:
        print(f"⚠ {host}의 robots.txt를 확인하지 못했어요. 사이트 이용약관을 직접 확인해 주세요")
    resp = http_get(url)
    if resp.encoding is None or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    try:
        tables = pd.read_html(io.StringIO(resp.text), match=match)
    except ValueError:
        raise CollectError(f"페이지에서 '{match}'가 들어간 표를 못 찾았어요. 표가 자바스크립트로 그려지는 페이지일 수 있어요") from None
    if index >= len(tables):
        raise CollectError(f"표가 {len(tables)}개뿐이에요. index를 0~{len(tables) - 1} 중에서 골라 주세요")
    table = tables[index]
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [" ".join(dict.fromkeys(str(c) for c in col)) for col in table.columns]
    return table


DRIVE_ROOT = "/content/drive/MyDrive/KFA_자료"
CARD_COLUMNS = ["파일명", "출처", "URL", "기준일", "신뢰도"]


# [5단계] 구글 드라이브 연결 후 KFA_자료 폴더 위치 돌려주기 (Colab 전용)
def mount_drive():
    try:
        from google.colab import drive
    except ImportError:
        raise CollectError("구글 드라이브 연결은 Colab에서만 돼요") from None
    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise CollectError("내 드라이브 맨 위에 'KFA_자료' 폴더가 없어요 (이름의 밑줄까지 똑같이)")
    return Path(DRIVE_ROOT)


# [5단계] 출처카드 읽기: CSV 경로 또는 구글 시트 제목
def read_source_cards(source="출처카드"):
    """A CSV path, or (in Colab) the title of a Google Sheet."""
    if str(source).lower().endswith(".csv") or Path(str(source)).exists():
        return read_table(source)
    try:
        from google.colab import auth
    except ImportError:
        raise CollectError("구글 시트는 Colab에서만 읽을 수 있어요. 여기서는 CSV 경로를 주세요") from None
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    client = gspread.authorize(creds)
    try:
        sheet = client.open(source).sheet1
    except gspread.SpreadsheetNotFound:
        files = client.list_spreadsheet_files()
        title = pick_sheet_title(source, [f["name"] for f in files])
        print(f"'{title}'를 {source}로 읽었어요")
        sheet = client.open_by_key(next(f["id"] for f in files if f["name"] == title)).sheet1
    return pd.DataFrame(sheet.get_all_records())


# [5단계] 시트 이름이 정확히 같지 않을 때 띄어쓰기만 다른 시트 찾기
def pick_sheet_title(wanted, titles):
    """The one title equal to wanted once all whitespace is removed; stops on zero or several."""
    squeeze = lambda s: re.sub(r"\s+", "", str(s))
    matches = [t for t in titles if squeeze(t) == squeeze(wanted)]
    if not matches:
        raise CollectError(f"'{wanted}'라는 구글 시트를 못 찾았어요. 시트 이름을 확인해 주세요")
    if len(matches) > 1:
        raise CollectError(f"'{wanted}'와 띄어쓰기만 다른 시트가 {len(matches)}개예요: {matches}. "
                           f"쓸 시트 하나만 이름을 '{wanted}'로 바꿔 주세요")
    return matches[0]


# [3·5단계] 출처카드에 적힌 파일만 읽어서 공통 표로. 못 읽는 파일은 이유와 함께 건너뜀
def collect_from_cards(cards, files_dir):
    """Read files listed on the source cards. Only CSV/XLSX already shaped as 항목/값/단위/기준일.

    An optional 공시일 card column fills 공시일 the same way 기준일 does: a cell in the file wins.
    """
    missing = [c for c in CARD_COLUMNS if c not in cards.columns]
    if missing:
        raise CollectError(f"출처카드에 없는 칸: {missing}")
    results, skipped = [], []
    for card in cards.to_dict("records"):
        name = str(card["파일명"]).strip()
        path = Path(files_dir) / name
        if not path.exists():
            skipped.append((name, "files/ 폴더에 이 이름의 파일이 없어요"))
            continue
        if path.suffix.lower() not in (".csv", ".xlsx"):
            skipped.append((name, "PDF·글 자료는 4회차에 AI가 읽어요"))
            continue
        table = read_table(path)
        if not {"항목", "값"} <= set(table.columns):
            skipped.append((name, "'항목·값·단위·기준일' 칸 모양이 아니에요. 3단계 예시처럼 items로 바꿔 주세요. "
                                  f"지금 칸: {[str(col) for col in table.columns]}"))
            continue
        level_match = re.match(r"\s*([1-5])", str(card.get("수집 방법") or ""))
        disclosed = None if is_blank(card.get("공시일")) else card["공시일"]
        results.append(collect_table(
            table, level=int(level_match.group(1)) if level_match else 5, source=card["출처"], source_url=card["URL"],
            reliability=card["신뢰도"], base_date=card["기준일"], disclosed=disclosed, note=f"출처카드: {name}",
        ))
    for name, why in skipped:
        print(f"   ⏭ {name}: {why}")
    if not results:
        raise SkipStep(f"출처카드 {len(cards)}줄 중 읽을 수 있는 숫자 파일이 없어요")
    return pd.concat(results, ignore_index=True)


# 키 읽기: Colab이면 Secrets, 내 컴퓨터면 .env. 값은 절대 출력하지 않음
def get_key(name):
    """Colab Secrets first, then .env / environment. Never print the value."""
    try:
        from google.colab import userdata
    except ImportError:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        return os.getenv(name) or None
    try:
        return userdata.get(name) or None
    except Exception:
        return None

STEP_RESULTS.clear()
print("✅ 공통 도구 준비 완료")

✅ 공통 도구 준비 완료


In [5]:
#@title ⚙️ 엔진 도구 (고치지 말고 실행만) { display-mode: "form" }
"""KFA AI agent study: report engine skeleton.

Adds row ids and a numbered source list on top of the shared record table from
collect.py, computes metrics from those rows, and checks the numbers written in
claims against them. Engine columns go on copies only; the 15 shared columns never
change. Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 안내] 이 셀은 고치지 않고 실행만 해요.
    with_ids()        공통 표의 줄마다 이름표(rid)를 붙여요. 같은 사실이 두 줄이면 멈춰요
    build_sources()   출처 목록 [S1] [S2] ...을 만들고, 줄마다 어느 출처인지(source_id) 붙여요
    compute_metrics() 지표 정의 표(METRIC_SPECS)대로 지표 [M1] [M2] ...을 계산해요. 입력 줄이 없으면 값을 비우고 이유를 적어요
    check_claims()    주장에 적은 숫자를 지표나 원본 줄과 대조해서 PASS / FIX / REVIEW로 판정해요
    render_check()    주장 목록·지표 표·숫자 대조 표·출처 목록을 마크다운 파일 하나로 저장해요
    claims_prompt()   Gemini에게 보낼 질문(규칙 + 공통 표 + 지표 표 + 글)을 만들어요
    ask_claims()      Gemini 답을 원문 그대로 저장해요. 질문이 저장된 것과 똑같으면 다시 부르지 않아요
    use_backup()      Gemini가 안 될 때, 질문이 똑같으면 스터디장이 미리 받아 둔 답(backup/)을 써요
    load_claims()     저장된 답을 주장 목록으로 읽어요. 깨진 주장은 REVIEW로 넘겨요
"""

import json
import re
import time
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Literal
from urllib.parse import quote

import pandas as pd


RID_PARTS = ["종목코드", "항목", "기준", "기준일", "출처"]
SOURCE_KEY = ["출처", "출처 URL"]
SOURCE_COLUMNS = ["source_id", "출처", "출처 URL", "수집 방법", "신뢰도", "기준일 범위", "줄 수"]
METRIC_COLUMNS = ["metric_id", "지표", "값", "단위", "계산식", "입력 rid", "source_id", "기준일", "상태", "이유"]
CHECK_COLUMNS = ["claim_id", "문장", "근거 id", "적은 숫자", "적은 단위", "실제 값", "판정", "이유", "출처 번호"]
METRIC_BASIS = "연결"
COMPUTED = "계산됨"
PASS, FIX, CHECK = "PASS", "FIX", "REVIEW"
UNITS = {
    "원": ("원", Decimal(1)),
    "천원": ("원", Decimal(10) ** 3),
    "백만원": ("원", Decimal(10) ** 6),
    "억원": ("원", Decimal(10) ** 8),
    "십억원": ("원", Decimal(10) ** 9),
    "조원": ("원", Decimal(10) ** 12),
    "%": ("%", Decimal(1)),
}


@dataclass(frozen=True)
class MetricSpec:
    """growth: inputs = (item,), compared with the same item one year earlier.
    ratio: inputs = (numerator item, denominator item), same date."""
    name: str
    unit: str
    formula: str
    kind: Literal["growth", "ratio"]
    inputs: tuple


METRIC_SPECS = (
    MetricSpec("매출 성장률", "%", "(올해 매출액(연간) − 작년 매출액(연간)) ÷ 작년 매출액(연간) × 100",
               "growth", ("매출액(연간)",)),
    MetricSpec("영업이익률", "%", "영업이익(연간) ÷ 매출액(연간) × 100", "ratio", ("영업이익(연간)", "매출액(연간)")),
    MetricSpec("부채비율", "%", "부채총계 ÷ 자본총계 × 100", "ratio", ("부채총계", "자본총계")),
)


def _text(value):
    return "" if is_blank(value) else str(value)


def _source_order(sid):
    return int(sid[1:])


def _require_tagged(records):
    missing = [col for col in ("rid", "source_id") if col not in records.columns]
    if missing:
        raise CollectError(f"공통 표에 {missing} 칸이 없어요. build_sources(with_ids(표)) 결과를 넣어 주세요")


def with_ids(records):
    """Copy with rid = 종목코드|항목|기준|기준일|출처; a repeated rid is the same fact twice and stops."""
    out = records.copy()
    out["rid"] = ["|".join(_text(v) for v in row) for row in out[RID_PARTS].itertuples(index=False)]
    repeated = out.loc[out["rid"].duplicated(keep=False), "rid"].unique().tolist()
    if repeated:
        raise CollectError(
            f"같은 사실(종목코드|항목|기준|기준일|출처)이 두 줄 이상 있어요: {repeated[:5]}. "
            "같은 자료를 두 번 붙였는지 확인해 주세요")
    return out


def build_sources(records):
    """Return (sources, records copy with source_id).

    One source = one (출처, 출처 URL) pair, so each DART filing keeps its own URL.
    Numbers follow the sorted pairs, not row order, so reordering cells keeps every S#.
    """
    out = records.copy()
    keys = [tuple(_text(v) for v in row) for row in out[SOURCE_KEY].itertuples(index=False)]
    ids = {key: f"S{n}" for n, key in enumerate(sorted(set(keys)), 1)}
    out["source_id"] = [ids[key] for key in keys]
    rows = []
    for (name, url), sid in ids.items():
        part = out[out["source_id"] == sid]
        row = {"source_id": sid, "출처": name, "출처 URL": url, "줄 수": len(part)}
        for col in ("수집 방법", "신뢰도"):
            values = part[col].map(_text).unique().tolist()
            if len(values) > 1:
                raise CollectError(f"[{sid}] {name}: 한 출처인데 {col}이(가) 여러 개예요 {values}. 하나로 맞춰 주세요")
            row[col] = values[0]
        dates = sorted(d for d in part["기준일"].map(_text) if d)
        if dates:
            row["기준일 범위"] = dates[0] if dates[0] == dates[-1] else f"{dates[0]} ~ {dates[-1]}"
        else:
            row["기준일 범위"] = ""
        rows.append(row)
    return pd.DataFrame(rows, columns=SOURCE_COLUMNS), out


def _find_input(records, stock_code, item, base_date):
    """Return (row, None) for exactly one usable consolidated row, else (row or None, reason)."""
    label = f"{item} {base_date}"
    same = records[(records["종목코드"].map(_text) == stock_code) & (records["항목"].map(_text) == item)
                   & (records["기준일"].map(_text) == base_date)]
    rows = same[same["기준"].map(_text) == METRIC_BASIS]
    if rows.empty:
        hint = " (다른 기준 줄만 있음)" if not same.empty else ""
        return None, f"{label}: 연결 줄 없음{hint}"
    if len(rows) > 1:
        return None, f"{label}: 연결 줄이 {len(rows)}개라 하나를 고를 수 없음 ({rows['출처'].tolist()})"
    row = rows.iloc[0]
    if row["상태"] == REVIEW:
        return row, f"{label}: 입력 줄이 확인 필요"
    if is_blank(row["값"]):
        return row, f"{label}: 값 없음"
    return row, None


def compute_metrics(records, specs, stock_code, base_date):
    """One row per spec, M# in spec order. A metric whose inputs are missing, flagged,
    ambiguous or have a denominator <= 0 keeps an empty value with 확인 필요 and the reasons."""
    _require_tagged(records)
    prev_date = f"{int(base_date[:4]) - 1}{base_date[4:]}"
    out = []
    for n, spec in enumerate(specs, 1):
        if spec.kind == "growth":
            wanted = [(spec.inputs[0], base_date), (spec.inputs[0], prev_date)]
        else:
            wanted = [(item, base_date) for item in spec.inputs]
        found, reasons = [], []
        for item, date in wanted:
            row, reason = _find_input(records, stock_code, item, date)
            if row is not None:
                found.append(row)
            if reason:
                reasons.append(reason)
        value = None
        if not reasons:
            first, second = (float(row["값"]) for row in found)
            if second <= 0:
                label = f"{wanted[1][0]} {wanted[1][1]}"
                if spec.kind == "growth":
                    reasons.append(f"기준값 {label}이(가) 0 이하(적자): 흑자전환 / 적자지속 / 적자확대로 표기")
                else:
                    reasons.append(f"분모 {label}이(가) 0 이하라 비율이 의미 없음")
            else:
                value = ((first - second) if spec.kind == "growth" else first) / second * 100
        out.append({
            "metric_id": f"M{n}", "지표": spec.name, "값": value, "단위": spec.unit, "계산식": spec.formula,
            "입력 rid": [row["rid"] for row in found],
            "source_id": sorted({row["source_id"] for row in found}, key=_source_order),
            "기준일": base_date, "상태": REVIEW if reasons else COMPUTED, "이유": "; ".join(reasons),
        })
    metrics = pd.DataFrame(out, columns=METRIC_COLUMNS)
    metrics["값"] = pd.Series([row["값"] for row in out], dtype=object)
    return metrics


def _written_number(number):
    """Exact decimal of the number as written, so '46.80' keeps its two decimals."""
    if isinstance(number, bool):
        return None
    try:
        value = Decimal(str(number).strip().replace(",", ""))
    except InvalidOperation:
        return None
    return value if value.is_finite() else None


def _unit_scales(unit, actual_unit):
    """Return ((written scale, actual scale), None) or (None, reason).

    Identical unit text compares 1:1 whatever the unit is, because the rid or M# already
    fixes what is measured. Blank units are refused, or two blanks would match as equal text.
    """
    written, source = "".join(_text(unit).split()), "".join(_text(actual_unit).split())
    if not written or not source:
        return None, f"단위 없음 (문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}')"
    if written == source:
        return (Decimal(1), Decimal(1)), None
    if written in UNITS and source in UNITS and UNITS[written][0] == UNITS[source][0]:
        return (UNITS[written][1], UNITS[source][1]), None
    return None, (f"단위가 안 맞음: 문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}' "
                  "(단위 글자가 같거나, 원·천원·백만원·억원·십억원·조원끼리만 바꿔 비교)")


def _check_one(number, unit, actual, actual_unit):
    """Return (actual in the written unit or None, verdict, reason)."""
    written = _written_number(number)
    if written is None:
        return None, CHECK, f"적은 숫자 '{number}'를 숫자로 못 읽음"
    scales, problem = _unit_scales(unit, actual_unit)
    if problem:
        return None, CHECK, problem
    scale, actual_scale = scales
    actual_base = Decimal(str(actual)) * actual_scale
    tolerance = Decimal("0.5").scaleb(written.as_tuple().exponent) * scale
    shown = float(actual_base / scale)
    if abs(written * scale - actual_base) <= tolerance:
        return shown, PASS, ""
    return shown, FIX, f"실제 값은 {_format_actual(shown, written, unit)}"


def _format_actual(actual, written, unit):
    """Two more decimals than the written number, so a FIX shows how far off it is."""
    places = max(-written.as_tuple().exponent, 0) + 2
    return f"{actual:,.{places}f} {unit}"


def _lookup(evidence_id, metrics, records):
    """Return (value, unit, source ids, reason the value cannot be used or None)."""
    if not evidence_id:
        return None, None, [], "근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음)"
    if evidence_id.startswith("M"):
        hit = metrics[metrics["metric_id"] == evidence_id]
        if hit.empty:
            return None, None, [], f"지표 {evidence_id}를 못 찾음"
        row = hit.iloc[0]
        if row["상태"] == REVIEW:
            return None, None, list(row["source_id"]), f"근거 지표가 확인 필요: {row['이유']}"
        return row["값"], row["단위"], list(row["source_id"]), None
    hit = records[records["rid"] == evidence_id]
    if hit.empty:
        return None, None, [], "근거 rid를 공통 표에서 못 찾음"
    row = hit.iloc[0]
    if row["상태"] == REVIEW:
        return None, None, [row["source_id"]], "근거 줄이 확인 필요"
    if is_blank(row["값"]):
        return None, None, [row["source_id"]], "근거 줄에 값 없음"
    return row["값"], _text(row["단위"]), [row["source_id"]], None


def check_claims(claims, metrics, records):
    """One row per (claim, evidence). Evidence ids starting with M are metrics, others are rids.

    claims: [{"claim_id": "C-1", "문장": "...", "근거": [(id, number, unit), ...]}].
    The number may be text ("46.80", "1,100") so trailing zeros keep their precision.
    A claim carrying "형식 오류" (set by load_claims) yields one REVIEW row and nothing else.
    출처 번호 collects the S# of every evidence row, or of a metric's input rows.
    """
    _require_tagged(records)
    out = []
    for claim in claims:
        evidence = claim.get("근거") or []
        rows, sources = [], set()
        problem = claim.get("형식 오류")
        if problem or not evidence:
            rows.append({"근거 id": "", "적은 숫자": None, "적은 단위": "", "실제 값": None, "판정": CHECK,
                         "이유": f"AI 답변 형식 오류: {problem}" if problem else "근거 없음"})
            evidence = []
        for evidence_id, number, unit in evidence:
            actual, actual_unit, sids, problem = _lookup(evidence_id, metrics, records)
            sources.update(sids)
            if problem:
                shown, verdict, reason = None, CHECK, problem
            else:
                shown, verdict, reason = _check_one(number, unit, actual, actual_unit)
            rows.append({"근거 id": evidence_id, "적은 숫자": number, "적은 단위": unit, "실제 값": shown,
                         "판정": verdict, "이유": reason})
        label = "".join(f"[{sid}]" for sid in sorted(sources, key=_source_order))
        for row in rows:
            out.append({"claim_id": claim.get("claim_id"), "문장": claim.get("문장"), **row, "출처 번호": label})
    return pd.DataFrame(out, columns=CHECK_COLUMNS)


def _cell(value):
    return _text(value).replace("|", "\\|").replace("\n", " ")


def _table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(_cell(v) for v in row) + " |" for row in rows]
    return lines


def _claim_number_rows(checked):
    for _, row in checked.iterrows():
        number, unit, actual = row["적은 숫자"], row["적은 단위"], row["실제 값"]
        written = _written_number(number)
        yield [
            f"{row['claim_id']} {row['문장']}",
            "" if is_blank(number) else f"{number} {unit}",
            "" if is_blank(actual) or written is None else _format_actual(actual, written, unit),
            f"{row['판정']}: {row['이유']}" if row["판정"] == CHECK else row["판정"],
        ]


def render_check(checked, metrics, sources, path):
    """Write claims, metrics, number checks and sources as one markdown file; return its path."""
    claims = checked.drop_duplicates("claim_id")
    lines = ["# 숫자 대조 결과", "", "## 주장", ""]
    lines += [f"- {row['claim_id']} {row['문장']} {row['출처 번호']}".rstrip() for _, row in claims.iterrows()]
    lines += ["", "## 지표", ""]
    lines += _table(["번호", "지표", "값", "계산식", "출처", "기준일"], [
        [row["metric_id"], row["지표"],
         f"확인 필요: {row['이유']}" if is_blank(row["값"]) else f"{row['값']:,.2f} {row['단위']}",
         row["계산식"], "".join(f"[{sid}]" for sid in row["source_id"]), row["기준일"]]
        for _, row in metrics.iterrows()])
    lines += ["", "## 숫자 대조", ""]
    lines += _table(["주장", "적은 숫자", "실제 값", "판정"], _claim_number_rows(checked))
    lines += ["", "## 출처", ""]
    lines += _table(["번호", "이름", "기준일 범위", "원본 URL", "신뢰도"], [
        [f"[{row['source_id']}]", row["출처"], row["기준일 범위"], row["출처 URL"], row["신뢰도"]]
        for _, row in sources.iterrows()])
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


BACKUP_URL = REPO_RAW_URL + "backup/"
RETRY_CODES = (429, 500, 503)
RETRY_WAITS = (5, 10, 20)
CLAIM_RULES = """아래 글에서 숫자가 들어간 주장을 모두 찾아, 각 숫자의 근거를 공통 표나 지표 표에서 골라 줘.

규칙
- 출력은 JSON 배열 하나만: [{"claim_id": "C-1", "문장": "...", "근거": [["근거 id", "숫자", "단위"]]}]
- claim_id는 C-1, C-2 ... 순서대로 붙여
- 문장은 글에 적힌 그대로 옮겨
- 근거 id는 공통 표의 rid나 지표 표의 metric_id(M1 ...)를 글자 그대로 복사해. 표에 없는 id를 만들지 마
- 숫자와 단위는 문장에 적힌 글자 그대로 적어 (예: "97.1", "조 원"). 단위를 바꾸거나 다시 계산하지 마
- 문장 속 숫자마다 근거를 하나씩 적어. 숫자가 가리키는 항목의 줄이나 지표를 골라. 값이 맞는지는 판단하지 마
- 증감, 포인트 차이 같은 변화량은 그 변화량 자체가 표에 있을 때만 고르고, 없으면 근거 id를 ""로 비워 (수준 지표에 변화량을 대지 마)
- 그 밖에는 항목 자체가 표에 없을 때만 근거 id를 ""로 비워
- 숫자가 없는 주장은 "근거": []로 적어"""


def claims_prompt(records, metrics, text):
    """Question text for Gemini: rules, the tables it may cite, then the text to read."""
    rows = records[["rid", "항목", "값", "단위", "기준일"]].to_csv(index=False, lineterminator="\n")
    metric_rows = metrics[["metric_id", "지표", "값", "단위"]].to_csv(index=False, lineterminator="\n")
    return f"{CLAIM_RULES}\n\n[공통 표]\n{rows}\n[지표 표]\n{metric_rows}\n[글]\n{text.strip()}\n"


def _call_gemini(prompt, api_key, model, fallback_model):
    """Return (answer text, model that answered). Same retry rule as setup/00_setup_check.ipynb:
    wait 5/10/20 s on 429/500/503, then try the fallback model once."""
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    config = types.GenerateContentConfig(response_mime_type="application/json", temperature=0)

    def ask(name):
        return client.models.generate_content(model=name, contents=prompt, config=config).text or ""

    try:
        for attempt, wait in enumerate((*RETRY_WAITS, None)):
            try:
                return ask(model), model
            except Exception as error:
                if getattr(error, "code", None) not in RETRY_CODES:
                    raise
                if wait is None:
                    break
                print(f"⏳ Gemini가 붐벼요. {wait}초 기다렸다가 다시 시도해요 ({attempt + 1}/{len(RETRY_WAITS)})")
                time.sleep(wait)
        print(f"⏳ 계속 안 돼서 예비 모델({fallback_model})로 한 번 더 시도해요")
        return ask(fallback_model), fallback_model
    except Exception as error:
        raise CollectError(_gemini_failure(getattr(error, "code", None), type(error).__name__)) from None


def _gemini_failure(code, error_name):
    """Member-facing reason a Gemini call failed. Only server-side trouble says 'try later';
    a key or model problem names what to fix instead."""
    if code in (400, 401, 403):
        return f"Gemini가 키를 받아 주지 않았어요 ({code}). Secrets의 GEMINI_API_KEY 값을 다시 확인해 주세요 (SETUP.md 9번)"
    if code == 404:
        return "Gemini 모델 이름을 찾을 수 없어요 (404). 스터디장에게 알려 주세요"
    if code in RETRY_CODES:
        return f"Gemini 서버가 계속 붐비거나 오류가 나요 ({code}). 내 설정 문제가 아니에요. 잠시 뒤 다시 실행해 주세요"
    return f"Gemini를 부르지 못했어요 ({code or error_name}). 인터넷 연결을 확인하고 잠시 뒤 다시 실행해 주세요"


def _record_path(path):
    return path.with_name(f"{path.stem}_기록.json")


def ask_claims(prompt, api_key, path, model, fallback_model, overwrite=False):
    """Save Gemini's raw answer at path and a record (question, model, time) beside it; return path.

    The saved answer is reused only when the saved question is identical to this one,
    so editing the text never checks an old answer. The record is removed before the
    answer is rewritten, so a record on disk always belongs to the answer next to it.
    """
    path = Path(path)
    record_path = _record_path(path)
    if not overwrite and path.exists() and record_path.exists():
        try:
            saved = json.loads(record_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            saved = {}
        if saved.get("질문") == prompt:
            print(f"♻ 같은 질문의 저장된 답을 다시 써요 ({saved.get('모델')}, {saved.get('호출 시각')})")
            return path
    if not api_key:
        raise SkipStep("Gemini 키가 없어요. Secrets에 GEMINI_API_KEY를 등록하고 노트북 액세스를 켠 뒤 다시 실행해 주세요 (SETUP.md 9번)")
    answer, used_model = _call_gemini(prompt, api_key, model, fallback_model)
    path.parent.mkdir(parents=True, exist_ok=True)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record = {"질문": prompt, "모델": used_model, "호출 시각": now_kst()}
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"✅ Gemini 답 저장: {path} ({used_model})")
    return path


def use_backup(prompt, path, name):
    """Fall back to the answer the study lead saved in backup/<name>.txt when Gemini fails.

    Used only when the backup's question is identical to this one, so a member who changed
    the text or the company never gets an answer to a different question. Returns path or None.
    """
    path = Path(path)
    try:
        record = json.loads(http_get(BACKUP_URL + quote(f"{name}_기록.json")).content.decode("utf-8"))
        if record.get("질문") != prompt:
            print("⏭ 미리 받아 둔 답은 기본 글·기본 종목용이라 지금 질문에는 쓸 수 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
            return None
        answer = http_get(BACKUP_URL + quote(f"{name}.txt")).content.decode("utf-8")
    except (CollectError, json.JSONDecodeError):
        print("⏭ 미리 받아 둔 답이 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
        return None
    record_path = _record_path(path)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"🗂 Gemini 호출이 안 돼서 스터디장이 미리 받아 둔 답을 썼어요 ({record.get('모델')}, {record.get('호출 시각')})")
    return path


def _salvage(text):
    """Split a broken answer into claim objects; a piece that does not parse stays a string.
    Claims hold no nested objects, so every '{' starts a new claim."""
    decoder, items = json.JSONDecoder(), []
    start = text.find("{")
    while start != -1:
        try:
            item, end = decoder.raw_decode(text, start)
        except json.JSONDecodeError:
            end = text.find("{", start + 1)
            items.append(text[start:end if end != -1 else len(text)])
            start = end
            continue
        items.append(item)
        start = text.find("{", end)
    return items


def _claim_from_answer(item, n, seen):
    if not isinstance(item, dict):
        found = re.search(r'"claim_id"\s*:\s*"([^"]+)"', item) if isinstance(item, str) else None
        claim_id = found.group(1) if found and found.group(1) not in seen else f"AI-{n}"
        seen.add(claim_id)
        return {"claim_id": claim_id, "문장": "", "근거": [],
                "형식 오류": f"JSON이 깨짐: {_text(item).strip()[:80]}"}
    problems = []
    claim_id = item.get("claim_id")
    if not isinstance(claim_id, str) or not claim_id.strip():
        problems.append("claim_id 없음")
        claim_id = f"AI-{n}"
    elif claim_id in seen:
        problems.append(f"claim_id {claim_id} 중복")
        claim_id = f"{claim_id} (AI-{n})"
    seen.add(claim_id)
    sentence = item.get("문장")
    if not isinstance(sentence, str):
        problems.append("문장 없음")
    evidence = item.get("근거")
    if not isinstance(evidence, list) or any(not isinstance(ev, list) or len(ev) != 3 for ev in evidence):
        problems.append("근거는 [근거 id, 숫자, 단위] 목록이어야 함")
        evidence = []
    claim = {"claim_id": claim_id, "문장": _text(sentence),
             "근거": [(_text(ev_id).strip(), number, _text(unit)) for ev_id, number, unit in evidence]}
    if problems:
        claim["형식 오류"] = "; ".join(problems)
    return claim


def load_claims(path):
    """Read a saved Gemini answer into check_claims input. Never raises on a bad answer:
    each claim that cannot be read becomes a claim carrying "형식 오류"."""
    text = Path(path).read_text(encoding="utf-8")
    body = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text)
    try:
        items = json.loads(body)
        items = items if isinstance(items, list) else [items]
    except json.JSONDecodeError:
        items = _salvage(body)
    seen = set()
    claims = [_claim_from_answer(item, n, seen) for n, item in enumerate(items, 1)]
    return claims or [{"claim_id": "AI-1", "문장": "", "근거": [], "형식 오류": "AI 답변에서 주장을 하나도 못 찾음"}]

print("✅ 엔진 도구 준비 완료")

✅ 엔진 도구 준비 완료


In [6]:
DART_API_KEY = get_key("DART_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")
print("DART 키:", "등록됨 ✅" if DART_API_KEY else "없음 ⏭ (Secrets에 DART_API_KEY + 노트북 액세스 켜기)")
print("Gemini 키:", "등록됨 ✅" if GEMINI_API_KEY else "없음 ⏭ (Secrets에 GEMINI_API_KEY)")

DART 키: 등록됨 ✅
Gemini 키: 등록됨 ✅


## 1. 밸류에이션·현재시점·검증 도구

과거 PER 차트는 **연말 종가+해당 연차 확정 실적의 사후 계산**임을 명시합니다(그 당시 이용 가능한 PER로 오인 금지). 현재 가치평가에서는 **최종 공시된 최신 TTM 실적과 현재 종가**를 사용합니다. EBITDA/DCF 입력은 DART 이름 검색값을 자동 승인하지 않습니다.


In [7]:
#@title 🧰 v2 밸류에이션 도구 { display-mode: "form" }
import json, re, os, math, base64, io
from pathlib import Path
from decimal import Decimal
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, HTML

FIN_KSIC_PREFIX = ("64", "65", "66")
MODEL_ORDER = ["PER", "PBR", "EV/EBITDA", "DCF", "PSR"]
STATUS_RANK = {"보류": 0, "조건부": 1, "적합": 2}

def fill(template, **kw):
    for k, v in kw.items():
        template = template.replace("{{" + k + "}}", str(v))
    left = re.findall(r"\{\{(\w+)\}", template)
    if left:
        print("⚠ 채워지지 않은 빈칸:", left)
    return template

def _num(x):
    v, _ = to_number(x)
    return v

def _norm_name(s):
    return re.sub(r"[\s·ㆍ\-\(\)\[\]/]", "", str(s or "")).lower()

def _dart_all(stock_code, year, api_key):
    if not api_key:
        raise SkipStep("DART 키가 없어요")
    corp_code, corp_name = dart_corp(stock_code, api_key)
    fs = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
        "reprt_code": DART_REPORT[4], "fs_div": "CFS"}).json()
    _dart_status(fs)
    return corp_code, corp_name, fs.get("list", [])

def _first_by_ids(rows, ids, sj=None):
    ids = set(ids)
    for r in rows:
        if r.get("account_id") in ids and (sj is None or r.get("sj_div") in sj):
            v = _num(r.get("thstrm_amount"))
            if v is not None:
                return v, r, "account_id"
    return None, None, None

def _match_by_name(rows, include_any, exclude=(), sj=None, sum_matches=False):
    hits = []
    for r in rows:
        if sj is not None and r.get("sj_div") not in sj:
            continue
        nm = _norm_name(r.get("account_nm"))
        if any(_norm_name(k) in nm for k in include_any) and not any(_norm_name(k) in nm for k in exclude):
            v = _num(r.get("thstrm_amount"))
            if v is not None:
                hits.append((v, r))
    if not hits:
        return None, [], None
    if sum_matches:
        # 동일 account_id가 중복되면 첫 값만 사용
        seen, vals, used = set(), [], []
        for v, r in hits:
            key = r.get("account_id") or r.get("account_nm")
            if key in seen:
                continue
            seen.add(key); vals.append(v); used.append(r)
        return sum(vals), used, "계정명 합산"
    return hits[0][0], [hits[0][1]], "계정명 근사"

def dart_val_core(stock_code, year, api_key):
    """한 해의 밸류에이션 핵심 데이터. exact/approx/missing 품질 꼬리표를 함께 반환."""
    corp_code, corp_name, rows = _dart_all(stock_code, year, api_key)
    out = {"종목코드": stock_code, "종목명": corp_name, "year": year, "quality": {}, "rcept_no": None}
    for r in rows:
        out["rcept_no"] = out["rcept_no"] or r.get("rcept_no")

    specs = {
        "revenue": (["ifrs-full_Revenue"], ("IS","CIS"), ["매출액","수익"], []),
        "op_income": (["dart_OperatingIncomeLoss"], ("IS","CIS"), ["영업이익"], []),
        "ni_owner": (["ifrs-full_ProfitLossAttributableToOwnersOfParent"], ("IS","CIS"), ["지배기업소유주지분순이익","지배주주순이익"], []),
        "eq_owner": (["ifrs-full_EquityAttributableToOwnersOfParent"], ("BS",), ["지배기업소유주지분","지배주주지분"], []),
        "cash": (["ifrs-full_CashAndCashEquivalents"], ("BS",), ["현금및현금성자산"], []),
        "cfo": (["ifrs-full_CashFlowsFromUsedInOperatingActivities"], ("CF",), ["영업활동현금흐름","영업활동으로인한현금흐름"], []),
    }
    for key, (ids, sj, names, exc) in specs.items():
        v, r, how = _first_by_ids(rows, ids, sj)
        if v is None:
            v, rs, how2 = _match_by_name(rows, names, exc, sj, False)
            r = rs[0] if rs else None
            how = how2
        out[key] = v
        out["quality"][key] = "exact" if how == "account_id" else ("approx" if v is not None else "missing")

    # 감가상각 + 무형자산상각: 현금흐름표에서 계정명 합산. 회사별 표기 차이가 커서 근사 꼬리표.
    da, da_rows, _ = _match_by_name(
        rows, ["감가상각비","감가상각","무형자산상각비","무형자산상각"],
        ["사용권자산처분","감가상각누계액"], ("CF",), True)
    out["da"] = da
    out["quality"]["da"] = "approx" if da is not None else "missing"

    # CAPEX: 유형/무형자산 취득 현금유출의 절대값 합. 자산 매각대금은 제외.
    capex_hits = []
    for r in rows:
        if r.get("sj_div") != "CF":
            continue
        nm = _norm_name(r.get("account_nm"))
        is_acq = any(x in nm for x in map(_norm_name, ["유형자산의취득","유형자산취득","무형자산의취득","무형자산취득","propertyplantandequipment","intangibleassets"]))
        is_sale = any(x in nm for x in map(_norm_name, ["처분","매각","disposal","proceeds"]))
        if is_acq and not is_sale:
            v = _num(r.get("thstrm_amount"))
            if v is not None:
                capex_hits.append((abs(v), r))
    if capex_hits:
        # account_id 중복 제거
        seen, vals = set(), []
        for v, r in capex_hits:
            key = r.get("account_id") or r.get("account_nm")
            if key not in seen:
                seen.add(key); vals.append(v)
        out["capex"] = sum(vals)
        out["quality"]["capex"] = "approx"
    else:
        out["capex"] = None
        out["quality"]["capex"] = "missing"

    # 이자부채: 단기/장기차입금, 사채, 리스부채. 상위 총계와 하위 상세를 동시에 더하지 않도록
    # 표준 ID 우선, 없을 때 계정명 근사. 결과는 보수적으로 'approx' 꼬리표.
    debt_patterns = ["단기차입금","장기차입금","유동성장기부채","유동성장기차입금","사채","리스부채","lease liabilities","borrowings","bonds"]
    debt_excl = ["매입채무","미지급금","충당부채","파생","전환권","상환할증금"]
    debt, debt_rows, _ = _match_by_name(rows, debt_patterns, debt_excl, ("BS",), True)
    out["debt"] = debt
    out["quality"]["debt"] = "approx" if debt is not None else "missing"

    # 보통주 유통주식수 / 업종
    sh = http_get(f"{DART_URL}/stockTotqySttus.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year), "reprt_code": DART_REPORT[4]}).json()
    out["shares"] = None
    if sh.get("status") == "000":
        for r in sh.get("list", []):
            if str(r.get("se","")).strip().startswith("보통주"):
                out["shares"] = _num(r.get("distb_stock_co")) or _num(r.get("istc_totqy"))
                break
    out["quality"]["shares"] = "exact" if out["shares"] else "missing"

    co = http_get(f"{DART_URL}/company.json", {"crtfc_key": api_key, "corp_code": corp_code}).json()
    out["induty"] = str(co.get("induty_code","") or "")
    return out

def krx_price_range(stock_code, start_year, end_year):
    import FinanceDataReader as fdr
    df = fdr.DataReader(stock_code, f"{start_year}-01-01", f"{end_year}-12-31")
    if df is None or len(df) == 0:
        raise CollectError(f"{stock_code}: KRX 주가를 못 받았어요")
    return df

def krx_price(stock_code, year):
    import FinanceDataReader as fdr
    df = fdr.DataReader(stock_code, f"{year}-01-01")
    if df is None or len(df) == 0:
        raise CollectError(f"{stock_code}: KRX 주가를 못 받았어요")
    now = (float(df.iloc[-1]["Close"]), df.index[-1].strftime("%Y-%m-%d"))
    ye_df = df[df.index <= f"{year}-12-31"]
    ye = (float(ye_df.iloc[-1]["Close"]), ye_df.index[-1].strftime("%Y-%m-%d")) if len(ye_df) else (None,None)
    return ye, now

def extra_records_v2(core_by_year, price_latest, stock_code):
    rows = []
    labels = [
        ("ni_owner","지배주주순이익(연간)","원"),("eq_owner","지배자본총계","원"),
        ("shares","보통주 유통주식수","주"),("cash","현금및현금성자산","원"),
        ("debt","이자부채(근사)","원"),("da","감가상각·무형자산상각(근사)","원"),
        ("cfo","영업활동현금흐름(연간)","원"),("capex","CAPEX(근사)","원"),
    ]
    for year,c in core_by_year.items():
        rcp = c.get("rcept_no") or ""
        url = f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcp}"
        for key,label,unit in labels:
            if c.get(key) is not None:
                q = c.get("quality",{}).get(key,"")
                rows.append({"항목":label,"값":c[key],"단위":unit,"기준일":f"{year}-12-31",
                             "공시일":rcp[:8],"출처 URL":url,"비고":f"추출품질={q}"})
    parts=[]
    if rows:
        t=pd.DataFrame(rows)
        parts.append(collect_table(t, level=1, source="DART 단일회사 전체 재무제표·주식총수 API",
                                   reliability="상", basis="연결", stock_code=stock_code,
                                   stock_name=next(iter(core_by_year.values()))["종목명"],
                                   source_url="https://opendart.fss.or.kr"))
    if price_latest and price_latest[0]:
        p=pd.DataFrame([{"항목":"주가(종가)","값":price_latest[0],"단위":"원",
                         "기준일":price_latest[1],"공시일":price_latest[1]}])
        parts.append(collect_table(p, level=2, source="KRX (FinanceDataReader)", reliability="중",
                                   basis=NOT_APPLICABLE, stock_code=stock_code,
                                   stock_name=next(iter(core_by_year.values()))["종목명"],
                                   source_url="https://data.krx.co.kr"))
    return pd.concat(parts,ignore_index=True) if parts else None

def item_value(table,item,year=None):
    t=table[table["항목"]==item]
    if year is not None:
        t=t[t["기준일"].astype(str).str.startswith(str(year))]
    return None if t.empty else float(t.sort_values("기준일")["값"].iloc[-1])

def _status(cap, reason, issues=None):
    return {"ceiling":cap,"reason":reason,"issues":issues or []}

def prescreen_v2(table, core, year):
    """5개 모형의 코드 상한. AI는 이보다 높은 판정을 낼 수 없다."""
    res={"is_financial":False,"model_caps":{},"facts":{},"issues":[]}
    c=core[year]
    ind=c.get("induty","")
    res["facts"]["업종코드(KSIC)"]=ind or "확인 불가"
    if ind.startswith(FIN_KSIC_PREFIX):
        res["is_financial"]=True
        res["issues"].append(f"금융업(업종코드 {ind}): 이 노트북 평가 범위 밖")
        for m in MODEL_ORDER: res["model_caps"][m]=_status("보류","금융업 평가 범위 밖")
        return res

    ni,eq,sh=c.get("ni_owner"),c.get("eq_owner"),c.get("shares")
    rev,op=c.get("revenue"),c.get("op_income")
    da,cash,debt=c.get("da"),c.get("cash"),c.get("debt")
    cfo,capex=c.get("cfo"),c.get("capex")
    eps=(ni/sh) if ni is not None and sh else None
    bps=(eq/sh) if eq is not None and sh else None
    ebitda=(op+da) if op is not None and da is not None else None
    net_debt=(debt-cash) if debt is not None and cash is not None else None
    res["facts"].update({
        "매출액(원)":rev,"영업이익(원)":op,"지배주주순이익(원)":ni,"지배자본(원)":eq,
        "유통주식수(주)":sh,"EPS(원)":eps,"BPS(원)":bps,"D&A(원)":da,"EBITDA(원)":ebitda,
        "현금(원)":cash,"이자부채(원)":debt,"순차입금(원)":net_debt,
        "영업현금흐름(원)":cfo,"CAPEX(원)":capex
    })

    # PER
    if eps is None or eps <= 0:
        res["model_caps"]["PER"]=_status("보류","EPS가 0 이하이거나 계산 불가")
    else:
        prev=core.get(year-1,{}).get("ni_owner")
        cap="적합"; issues=[]
        if prev is None or prev <= 0 or (prev and (ni/prev > 2 or ni/prev < 0.5)):
            cap="조건부"; issues.append("순이익의 연속성/대표성 점검 필요")
        res["model_caps"]["PER"]=_status(cap,"양의 EPS",issues)

    # PBR
    if eq is None or eq <= 0 or bps is None:
        res["model_caps"]["PBR"]=_status("보류","자기자본이 0 이하이거나 계산 불가")
    else:
        res["model_caps"]["PBR"]=_status("적합","양의 지배자본과 BPS 확보")

    # EV/EBITDA
    need_ev=[sh,ebitda,net_debt]
    if any(x is None for x in need_ev) or ebitda <= 0:
        res["model_caps"]["EV/EBITDA"]=_status("보류","양의 EBITDA·순차입금·주식수 중 부족")
    else:
        q=[c.get("quality",{}).get(k) for k in ("da","debt","cash")]
        cap="조건부" if any(x=="approx" for x in q) else "적합"
        res["model_caps"]["EV/EBITDA"]=_status(cap,"EBITDA와 순차입금 계산 가능",
                                               ["D&A/부채 계정명 근사 포함"] if cap=="조건부" else [])

    # DCF
    req={"매출":rev,"영업이익":op,"D&A":da,"CAPEX":capex,"순차입금":net_debt,"주식수":sh}
    missing=[k for k,v in req.items() if v is None]
    if missing:
        res["model_caps"]["DCF"]=_status("보류","DCF 입력 부족: "+", ".join(missing))
    else:
        q=[c.get("quality",{}).get(k) for k in ("da","capex","debt","cash")]
        cap="조건부" if any(x=="approx" for x in q) else "적합"
        res["model_caps"]["DCF"]=_status(cap,"5년 단순 FCFF 계산 입력 확보",
                                         ["ΔNWC=0 가정","D&A/CAPEX/부채 계정 근사 가능"] if cap=="조건부" else ["ΔNWC=0 가정"])

    # PSR
    if rev is None or rev <= 0 or not sh:
        res["model_caps"]["PSR"]=_status("보류","양의 매출·주식수 부족")
    else:
        res["model_caps"]["PSR"]=_status("적합","양의 매출과 주식수 확보")

    bad=table[table["상태"]=="확인 필요"] if table is not None else pd.DataFrame()
    if len(bad): res["issues"].append(f"'확인 필요' 줄 {len(bad)}개")
    return res

def apply_ai_caps(code_caps, ai_rows):
    """AI 판정을 코드 상한과 비교. 더 높으면 코드 상한으로 강제."""
    out=[]
    by={r.get("model"):r for r in (ai_rows or [])}
    for m in MODEL_ORDER:
        ceiling=code_caps[m]["ceiling"]
        ai=by.get(m,{})
        proposed=ai.get("status",ceiling)
        if proposed not in STATUS_RANK: proposed=ceiling
        final = proposed if STATUS_RANK[proposed] <= STATUS_RANK[ceiling] else ceiling
        out.append({"model":m,"code_ceiling":ceiling,"ai_status":proposed,"final_status":final,
                    "code_reason":code_caps[m]["reason"],"ai_reason":ai.get("reason","")})
    return pd.DataFrame(out)

# Gemini 호출: 일시적인 오류만 자동 재시도하며 잘못된 키/모델/영구 할당량은 숨기지 않습니다.
# 주의: 동일 프롬프트 재시도는 API 과금/호출량을 늘릴 수 있습니다.
RETRY_TEXT_CODES = (429, 500, 502, 503, 504)

def _gemini_error_code(error):
    code = getattr(error, "code", None)
    if hasattr(code, "value"):
        code = code.value
    try:
        return int(code)
    except (TypeError, ValueError):
        text = str(error)
        match = re.search(r"(?:^|\b)(429|500|502|503|504)(?:\b|$)", text)
        return int(match.group(1)) if match else None

def _gemini_error_detail(error):
    # 개인 API 키·요청 원문이 로그에 노출되지 않도록 오류 일부만 추출
    return str(error).replace('\n', ' ')[:350]

def _gemini_permanent_quota(error):
    """429라도 무료 티어 quota=0/일일 한도 소진인 경우 기다려도 해결되지 않을 수 있습니다."""
    detail = str(error).lower()
    hard_markers = (
        'limit: 0', 'limit:0', 'quota_value: 0', 'quota_value:0',
        'quota value: 0', 'quota exceeded for metric', 'billing required',
        'billing not enabled', 'free tier is not available',
    )
    # quota exceeded 라는 단어만으로 영구 문제라고 판단하지 않습니다.
    return any(marker in detail for marker in hard_markers)

def _gemini_retry_sleep(seconds):
    """길게 대기해도 Colab의 중지 버튼/KeyboardInterrupt가 빠르게 반영되도록 분할 sleep."""
    import time
    remaining = float(seconds)
    while remaining > 0:
        chunk = min(remaining, 2.0)
        time.sleep(chunk)
        remaining -= chunk

def _gemini_call(prompt, json_mode):
    """일시 장애 시 모델들을 순환하며 자동 재시도. 정상 응답 전까지 셀 종료 안 함.

    401/403/404/400 등 비재시도 오류, 영구 quota, Colab 중지 명령은 즉시 중단.
    로컬 Colab 세션이 끊기면 이 Python 실행 자체가 종료됩니다.
    """
    if not GEMINI_API_KEY:
        raise CollectError('GEMINI_API_KEY가 없습니다. Colab Secrets 접근 권한을 확인하세요')
    import time
    from google import genai
    from google.genai import types
    options = types.HttpOptions(timeout=GEMINI_REQUEST_TIMEOUT_MS)
    client = genai.Client(api_key=GEMINI_API_KEY, http_options=options)
    cfg = types.GenerateContentConfig(
        temperature=0 if json_mode else 0.2,
        **({'response_mime_type': 'application/json'} if json_mode else {})
    )
    models = list(dict.fromkeys((GEMINI_MODEL, GEMINI_FALLBACK_MODEL)))
    models = [m for m in models if m]
    if not models:
        raise CollectError('Gemini 모델명이 설정되지 않았습니다')

    attempt = 0
    started = time.monotonic()
    backoff = float(GEMINI_BACKOFF_INITIAL_SECONDS)
    retry_forever = bool(GEMINI_RETRY_FOREVER)
    cap = float(GEMINI_BACKOFF_MAX_SECONDS)
    max_total_wait = float(GEMINI_MAX_TOTAL_WAIT_SECONDS)
    while True:
        model_name = models[attempt % len(models)]
        attempt += 1
        try:
            response = client.models.generate_content(model=model_name, contents=prompt, config=cfg)
            result = (response.text or '').strip()
            if not result:
                raise CollectError(f'{model_name}에서 빈 응답을 받았습니다. 검증 없이 재생성하지 않습니다')
            print(f'✅ Gemini 응답 완료: {model_name} (총 호출 시도 {attempt}회)')
            return result
        except KeyboardInterrupt:
            print('⏹ 사용자가 Gemini 자동 재시도를 중지했습니다.')
            raise
        except CollectError:
            raise
        except Exception as e:
            code = _gemini_error_code(e)
            # HTTP 코드 없는 일시적 네트워크 타임아웃도 서버 혼잡 시 자주 발생
            exception_name = type(e).__name__.lower()
            detail_lower = str(e).lower()
            timeout_or_connect = (isinstance(e, (TimeoutError, ConnectionError))
                                  or any(x in exception_name for x in ('timeout', 'connection', 'connecterror'))
                                  or any(x in detail_lower for x in ('timed out', 'connection reset', 'temporarily unavailable')))
            if code not in RETRY_TEXT_CODES and not timeout_or_connect:
                raise CollectError(
                    f'Gemini 비재시도 오류 ({model_name}, HTTP {code or type(e).__name__}): '
                    f'{_gemini_error_detail(e)}'
                ) from None
            if code == 429 and _gemini_permanent_quota(e):
                raise CollectError(
                    'Gemini 429: 현재 프로젝트/모델의 할당량 또는 결제 설정 문제입니다. '
                    '일시적 혼잡으로 취급하지 않고 중단합니다. '
                    + _gemini_error_detail(e)
                ) from None
            elapsed = time.monotonic() - started
            if not retry_forever or (max_total_wait > 0 and elapsed >= max_total_wait):
                raise CollectError(
                    f'Gemini 일시 오류 재시도 제한 도달 (HTTP {code}, 총 {attempt}회, {elapsed:.0f}초).'
                ) from None
            # 두 모델이 모두 오류이면 지수적으로 백오프; 모델 교대 자체는 빠르게 수행
            completed_round = attempt % len(models) == 0
            delay = max(0.0, min(backoff, cap)) if completed_round else 2.0
            if completed_round:
                backoff = min(cap, max(1.0, backoff) * 2)
            if max_total_wait > 0:
                delay = min(delay, max_total_wait - elapsed)
            print(
                f'⏳ Gemini 일시 오류 HTTP {code} ({model_name}) · {attempt}회 실패 · '
                f'다음 시도 {delay:g}초 후 · 다음 모델 {models[attempt % len(models)]} · '
                f'총 경과 {elapsed:.0f}초 (Colab ■로 중단 가능)', flush=True
            )
            _gemini_retry_sleep(delay)

def ask_json(prompt): return _gemini_call(prompt, True)
def ask_text(prompt): return _gemini_call(prompt, False)

def parse_json_obj(text):
    body=re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$","",text.strip())
    try: return json.loads(body),None
    except Exception:
        m=re.search(r"\{.*\}",body,re.S)
        try: return json.loads(m.group(0)),None
        except Exception as e: return None,f"JSON 읽기 실패: {e}"

def multiples_from_core(c, price):
    sh=c.get("shares"); ni=c.get("ni_owner"); eq=c.get("eq_owner"); rev=c.get("revenue")
    da=c.get("da"); op=c.get("op_income"); cash=c.get("cash"); debt=c.get("debt")
    if not sh or not price: raise CollectError("주식수 또는 주가 없음")
    market_cap=price*sh
    eps=ni/sh if ni is not None else None
    bps=eq/sh if eq is not None else None
    ebitda=(op+da) if op is not None and da is not None else None
    net_debt=(debt-cash) if debt is not None and cash is not None else None
    ev=(market_cap+net_debt) if net_debt is not None else None
    return {
        "PER": price/eps if eps and eps>0 else None,
        "PBR": price/bps if bps and bps>0 else None,
        "EV/EBITDA": ev/ebitda if ev is not None and ebitda and ebitda>0 else None,
        "PSR": market_cap/rev if rev and rev>0 else None,
        "시가총액":market_cap,"EV":ev,"EBITDA":ebitda,"순차입금":net_debt
    }

def project_financials(a, base, years=5):
    rev0=base["revenue"]; rows=[]
    rev=rev0
    for y in range(1,years+1):
        rev*=1+float(a["growth"])
        op=rev*float(a["op_margin"])
        da=rev*float(a["da_to_revenue"])
        capex=rev*float(a["capex_to_revenue"])
        tax=float(a["tax_rate"])
        fcff=op*(1-tax)+da-capex  # ΔNWC = 0
        ni=op*float(a["net_to_op"])
        rows.append({"year":y,"revenue":rev,"op":op,"da":da,"capex":capex,"fcff":fcff,"ni":ni})
    return pd.DataFrame(rows)

def fair_value_v2(model,a,base,years=5):
    """상대가치(PER/PBR/EV/EBITDA/PSR)는 1년 선행, DCF만 5년 명시 추정."""
    proj=project_financials(a,base,years)
    y1=proj.iloc[0]; last=proj.iloc[-1]; shares=base["shares"]; net_debt=base["net_debt"]
    if model=="PER":
        eps=y1["ni"]/shares
        value=eps*float(a["target_per"])
        detail={"metric":eps,"multiple":a["target_per"],"평가기준":"1년 선행"}
    elif model=="PBR":
        retained=y1["ni"]*(1-float(a["payout"]))
        bps=(base["equity"]+retained)/shares
        value=bps*float(a["target_pbr"])
        detail={"metric":bps,"multiple":a["target_pbr"],"평가기준":"1년 선행"}
    elif model=="EV/EBITDA":
        ebitda=y1["op"]+y1["da"]
        ev=ebitda*float(a["target_ev_ebitda"])
        value=(ev-net_debt)/shares
        detail={"metric":ebitda,"multiple":a["target_ev_ebitda"],"EV":ev,"평가기준":"1년 선행"}
    elif model=="PSR":
        market_cap=y1["revenue"]*float(a["target_psr"])
        value=market_cap/shares
        detail={"metric":y1["revenue"],"multiple":a["target_psr"],"평가기준":"1년 선행"}
    elif model=="DCF":
        wacc=float(a["wacc"]); g=float(a["terminal_growth"])
        if wacc <= g: raise ValueError("DCF는 WACC > 영구성장률이어야 합니다")
        pv=sum(float(r["fcff"])/((1+wacc)**int(r["year"])) for _,r in proj.iterrows())
        tv=float(last["fcff"])*(1+g)/(wacc-g)
        pv_tv=tv/((1+wacc)**years)
        ev=pv+pv_tv
        value=(ev-net_debt)/shares
        detail={"PV_FCFF":pv,"PV_TV":pv_tv,"EV":ev,"wacc":wacc,"g":g,"평가기준":f"{years}년 DCF"}
    else:
        raise ValueError(model)
    return {"주당가치(원)":float(value),"예상매출(억원)":float(y1["revenue"]/1e8),
            "예상영업이익(억원)":float(y1["op"]/1e8),"예상순이익(억원)":float(y1["ni"]/1e8),**detail}

def model_extreme_flag(value, current):
    if value is None or not np.isfinite(value) or current in (None,0): return "계산 불가"
    ratio=value/current
    return "경고" if ratio < EXTREME_VALUE_LOW or ratio > EXTREME_VALUE_HIGH else "정상"

def make_per_chart(hist_df, path):
    import matplotlib.pyplot as plt
    d=hist_df.dropna(subset=["PER"]).copy()
    if d.empty: return None
    fig,ax=plt.subplots(figsize=(7,3.2))
    ax.plot(d["연도"],d["PER"],marker="o")
    ax.set_title("Retrospective FY PER (year-end close / later-reported earnings)")
    ax.set_xlabel("Fiscal year"); ax.set_ylabel("PER (x)"); ax.grid(True,alpha=.25)
    fig.tight_layout(); fig.savefig(path,dpi=150,bbox_inches="tight"); plt.close(fig)
    return path

def html_escape(x):
    import html
    return html.escape(str(x))

def df_html(df, digits=2):
    if df is None or len(df)==0: return "<p>데이터 없음</p>"
    d=df.copy()
    for c in d.select_dtypes(include="number").columns:
        d[c]=d[c].map(lambda x: "" if pd.isna(x) else f"{x:,.{digits}f}")
    return d.to_html(index=False,escape=True,border=0,classes="data")

def build_html_report(company, current_price, model_table, actuals, forecast, calc_long,
                      hist, peers, limitations, ai_summary, chart_path, out_path):
    img=""
    if chart_path and os.path.exists(chart_path):
        b64=base64.b64encode(open(chart_path,"rb").read()).decode()
        img=f'<img class="chart" src="data:image/png;base64,{b64}"/>'
    lim="".join(f"<li>{html_escape(x)}</li>" for x in limitations if x)
    css="""body{font-family:Arial,'Malgun Gothic',sans-serif;margin:30px;color:#222;max-width:1180px}
    h1{margin-bottom:4px}h2{border-bottom:2px solid #222;padding-bottom:6px;margin-top:28px}
    .muted{color:#666}.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}
    .card{border:1px solid #ddd;border-radius:10px;padding:16px}.data{border-collapse:collapse;width:100%;font-size:13px}
    .data th,.data td{border-bottom:1px solid #e5e5e5;padding:7px;text-align:right}.data th:first-child,.data td:first-child{text-align:left}
    .chart{max-width:760px;width:100%}.warn{background:#fff6e5;padding:12px;border-radius:8px}"""
    body=f"""<!doctype html><html><head><meta charset="utf-8"><style>{css}</style></head><body>
    <h1>{html_escape(company)} 밸류에이션 브리프</h1>
    <div class="muted">최근 확인 가능한 종가 {current_price:,.0f}원 ({html_escape(VALUATION_PRICE[1])}) · 공시실적: {html_escape(core[YEAR].get('valuation_period','FY'))} / {html_escape(core[YEAR].get('valuation_period_end',''))} · 공시접수 {html_escape(core[YEAR].get('valuation_filing_dt',''))} · Python 계산 / Gemini 서술 · 투자 권유 아님</div>
    <h2>1. 평가 모형 적합성</h2>{df_html(model_table)}
    <h2>2. 확정 실적</h2>{df_html(actuals)}
    <h2>3. 매출·이익 추정</h2>{df_html(forecast)}
    <h2>4. 과거 연말 기준 사후 배수(현재 평가와 구분)</h2><p class="muted">확정 결산 실적은 해당 연말에 미공개일 수 있으므로 역사적 투자 시점의 PER이 아닙니다.</p>{df_html(hist)}{img}
    <h2>5. 동종기업 비교</h2>{df_html(peers)}
    <h2>6. 시나리오별 가치평가</h2>{df_html(calc_long)}
    <h2>7. 주력·보조 모형 해석</h2><div class="card">{html_escape(ai_summary)}</div>
    <h2>8. 민감도·검증 경고</h2><div class="warn">극단값 검사는 현재가의 {EXTREME_VALUE_LOW:.1f}배~{EXTREME_VALUE_HIGH:.1f}배 밖을 경고합니다. 경고는 오류 확정이 아니라 재검토 신호입니다.</div>
    <h2>9. 한계 및 확인 필요</h2><ul>{lim}</ul>
    </body></html>"""
    Path(out_path).write_text(body,encoding="utf-8")
    return out_path

print("✅ v2 밸류에이션 도구 준비 완료")

# ===== v3: 엄격 추출·근거 있는 수동 확정 =====
DART_CORE_V2_UNSAFE = dart_val_core

def vetted_account_override(c, key, entry, year):
    if not isinstance(entry,dict): raise ValueError(f"{key}: 근거 메타데이터가 필요합니다")
    require=("value","source_url","basis","asof","account_note")
    if any(not entry.get(k) and entry.get(k)!=0 for k in require):
        raise ValueError(f"{key}: 필수항목 누락 {require}")
    if entry["basis"] != "연결" or entry["asof"] != f"{year}-12-31":
        raise ValueError(f"{key}: 연결·기말 기준 확인 필요")
    if not str(entry["source_url"]).startswith("https://dart.fss.or.kr/"):
        raise ValueError(f"{key}: DART 원문 URL을 입력해야 합니다")
    v=float(entry["value"])
    if not math.isfinite(v) or v<0: raise ValueError(f"{key}: 유한한 0 이상 값이어야 합니다")
    if len(str(entry["account_note"]).strip())<12: raise ValueError(f"{key}: 합계/중복 검증 근거를 자세히 적어주세요")
    c[key]=v; c["quality"][key]="reviewed"
    c.setdefault("lineage",{})[key]={k:entry[k] for k in require}

def dart_val_core(stock_code, year, api_key):
    """애매한 합산 추출은 폐기. 승인된 원문 수치만 후속 평가에 투입."""
    c=DART_CORE_V2_UNSAFE(stock_code,year,api_key)
    # v2 name-match 기반 합산은 신뢰할 수 없으므로 폐기
    for key in ("da","capex","debt"):
        c[key]=None; c["quality"][key]="missing"
    # 핵심 단일 계정명 근사도 확정 실적으로 사용 금지
    for key in ("revenue","op_income","ni_owner","eq_owner","cash","cfo"):
        if c["quality"].get(key)!="exact":
            c[key]=None; c["quality"][key]="missing"
    # 연결 종목 및 연도 확인, 수동 데이터는 분석기업에만 적용
    if stock_code==STOCK_CODE and year==YEAR:
        for key,entry in VERIFIED_FINANCIAL_OVERRIDES.items():
            if key not in {"da","capex","debt"}: raise ValueError(f"허용되지 않는 override: {key}")
            vetted_account_override(c,key,entry,year)
    return c

# 수동확정 재무제표에 정확히 기재한 자료만 'reviewed' 처리. 공시 코드만으로 추정하지 않음.

def validate_market_basis(c, price_tuple, year):
    if not price_tuple or len(price_tuple)<2 or price_tuple[0] is None:
        raise ValueError("연말 주가가 없습니다")
    price, dt=price_tuple
    dt=str(dt)[:10]
    # 휴장일 고려해 마지막 거래일은 12월 중으로 확인
    if not dt.startswith(f"{year}-12-"):
        raise ValueError(f"가격 기준일 불일치: {dt} != {year}년 12월")
    if not (c.get("shares") and c["shares"]>0):
        raise ValueError("기말 보통주 주식수가 없습니다")
    if not float(price)>0: raise ValueError("기준 주가가 양수가 아닙니다")
    return float(price),dt

def peer_reference_bands(peers):
    bands={}
    for m in ("PER","PBR","EV/EBITDA","PSR"):
        if peers is None or m not in peers: continue
        vals=pd.to_numeric(peers[m],errors="coerce")
        vals=vals[(vals>0)&np.isfinite(vals)]
        # 최소 두 개 비교기업, 1개일 때는 검증할 기준 없음
        if len(vals)>=2:
            med=float(vals.median()); bands[m]=(PEER_MULT_LOWER*med,PEER_MULT_UPPER*med,med)
    return bands

# 필수 필드는 모델의 실제 사용 여부에 따라 달라진다. 보류 모형의 배수는 null 허용.
MODEL_TARGET_KEYS = {"PER":"target_per", "PBR":"target_pbr", "EV/EBITDA":"target_ev_ebitda", "PSR":"target_psr"}
BASE_REQUIRED_KEYS = ("growth", "op_margin", "net_to_op")
OPTIONAL_DEFAULTS = {"payout":0.0, "tax_rate":0.0, "da_to_revenue":0.0,
                     "capex_to_revenue":0.0}

def scenario_required_keys(models):
    keys=set(BASE_REQUIRED_KEYS)
    if "PBR" in models: keys.add("payout")
    if "EV/EBITDA" in models or "DCF" in models: keys.add("da_to_revenue")
    if "DCF" in models: keys.update(("tax_rate","capex_to_revenue","wacc","terminal_growth"))
    for model, field in MODEL_TARGET_KEYS.items():
        if model in models: keys.add(field)
    return keys

def normalize_scenarios_v4(result, models):
    """검증 전 비사용 필드만 계산용 중립값으로 채운다. 사용 모형 필드는 절대 자동 생성하지 않는다."""
    if not isinstance(result,dict): return result
    sc=result.get("scenarios")
    if not isinstance(sc,dict): return result
    required=scenario_required_keys(models)
    for label in ("bear","base","bull"):
        a=sc.get(label)
        if not isinstance(a,dict): continue
        for field, default in OPTIONAL_DEFAULTS.items():
            if field not in required and (field not in a or a[field] is None):
                a[field] = default
        for field in MODEL_TARGET_KEYS.values():
            if field not in required and field not in a: a[field]=None
    return result

def validate_scenarios_v3(step2, core_now, bands, selected_models, years=5):
    """v4 검증: 사용 중인 모형만 엄격한 필수 검사, 경제적 순서/동종 배수 검사 유지."""
    errs=[]
    sc=step2.get("scenarios",{}) if isinstance(step2,dict) else {}
    if not isinstance(sc,dict) or set(sc)!={"bear","base","bull"}:
        return ["세 시나리오(bear/base/bull)가 정확히 필요합니다"]
    required=scenario_required_keys(selected_models)
    numeric={}
    for label in ("bear","base","bull"):
        a=sc[label]
        if not isinstance(a,dict): errs.append(f"{label}: dict 아님"); continue
        numeric[label]={}
        for field in required:
            try:
                val=float(a[field])
                if not math.isfinite(val): raise ValueError()
                numeric[label][field]=val
            except (KeyError,TypeError,ValueError):
                errs.append(f"{label}.{field} 값이 없거나 유한숫자가 아님: {repr(a.get(field))[:100]}")
    if errs: return errs
    bounds={"growth":(-.60,.80),"op_margin":(-.20,.60),"net_to_op":(0,2),
            "payout":(0,1),"tax_rate":(0,.50),"da_to_revenue":(0,.30),
            "capex_to_revenue":(0,.50),"wacc":(.04,.30),"terminal_growth":(-.02,.04)}
    for label,a in numeric.items():
        for field,(low,high) in bounds.items():
            if field in required and not low<=a[field]<=high:
                errs.append(f"{label}.{field}: 범위 {low}~{high} 위반 ({a[field]})")
        if "DCF" in selected_models and a["wacc"]-a["terminal_growth"] < .02:
            errs.append(f"{label}: WACC-g 2%p 이상 필요")
        if "EV/EBITDA" in selected_models and a["da_to_revenue"]+a["op_margin"]<=0:
            errs.append(f"{label}: EBITDA가 양수여야 함")
        for model, field in MODEL_TARGET_KEYS.items():
            if model not in selected_models: continue
            if model not in bands: errs.append(f"{model}: 비교기업 유효 배수 2개 이상 필요")
            elif not bands[model][0]<=a[field]<=bands[model][1]:
                errs.append(f"{label}.{field}: 동종기업 중앙값 기반 허용 범위 밖")
    if errs: return errs
    b={"revenue":core_now.get("revenue"),"equity":core_now.get("eq_owner"),
       "shares":core_now.get("shares"),
       "net_debt":(core_now["debt"]-core_now["cash"] if core_now.get("debt") is not None
                    and core_now.get("cash") is not None else None)}
    if b["revenue"] is None or not b["shares"]: return ["기초 실적 누락"]
    proj={}
    for label in ("bear","base","bull"):
        try: proj[label]=project_financials(sc[label], b, years).iloc[0]
        except Exception as e: errs.append(f"{label} 예상실적 계산 실패: {type(e).__name__}: {e}")
    if len(proj)==3:
        for field in ("revenue","op","ni"):
            values=[float(proj[k][field]) for k in ("bear","base","bull")]
            if not values[0]<=values[1]<=values[2]: errs.append(f"{field}: Bear ≤ Base ≤ Bull 위반")
        for model in selected_models:
            try:
                values=[fair_value_v2(model,sc[k],b,years)["주당가치(원)"] for k in ("bear","base","bull")]
                if not all(math.isfinite(v) and v>0 for v in values): errs.append(f"{model}: 주당가치 비양수/비유한")
                elif not values[0]<=values[1]<=values[2]: errs.append(f"{model}: Bear ≤ Base ≤ Bull 위반")
            except Exception as e: errs.append(f"{model}: 가치계산 실패 {type(e).__name__}: {e}")
    notes=step2.get("assumption_notes",[]) if isinstance(step2,dict) else []
    if not isinstance(notes,list) or not any(isinstance(r,dict) and r.get("item")=="growth" and str(r.get("rationale", "")).strip() for r in notes):
        errs.append("성장률 가정 근거 없음")
    return errs

def independent_fair_value_check(model,a,base,years):
    """독립 수식 경로: 원래 fair_value_v2를 재호출하지 않는다."""
    s=float(base["shares"]);rev=float(base["revenue"]);eq=base.get("equity")
    nd=base.get("net_debt");f=lambda k:float(a[k])
    rev1=rev*(1+f("growth"));op1=rev1*f("op_margin");ni1=op1*f("net_to_op")
    if model=="PER":return ni1*f("target_per")/s
    if model=="PBR":return (float(eq)+ni1*(1-f("payout")))*f("target_pbr")/s
    if model=="PSR":return rev1*f("target_psr")/s
    if model=="EV/EBITDA":return ((op1+rev1*f("da_to_revenue"))*f("target_ev_ebitda")-float(nd))/s
    if model=="DCF":
        r=rev;pv=0
        for t in range(1,years+1):
            r=r*(1+f("growth"))
            cf=r*(f("op_margin")*(1-f("tax_rate"))+f("da_to_revenue")-f("capex_to_revenue"))
            pv+=cf/(1+f("wacc"))**t
        pv+=cf*(1+f("terminal_growth"))/(f("wacc")-f("terminal_growth"))/(1+f("wacc"))**years
        return (pv-float(nd))/s
    raise ValueError(model)

def dcf_sensitivity(a,base,years=5):
    rows=[]
    for dw in (-.01,0,.01):
        for dg in (-.005,0,.005):
            aa=dict(a);aa["wacc"]=float(a["wacc"])+dw
            aa["terminal_growth"]=float(a["terminal_growth"])+dg
            if aa["wacc"]-aa["terminal_growth"]<.02: val=np.nan
            else:val=independent_fair_value_check("DCF",aa,base,years)
            rows.append({"WACC":aa["wacc"],"영구성장률":aa["terminal_growth"],"주당가치(원)":val})
    return pd.DataFrame(rows)


# ===== v5: 현재 시점·공시 선행성·TTM/상위분석 인계 =====
import copy
from datetime import datetime, timedelta, date
from zoneinfo import ZoneInfo
import calendar

ASOF_DATE = datetime.now(ZoneInfo("Asia/Seoul")).date()

# 입력 시점은 고정값이 아니라 실행 시각의 현지 날짜입니다.
def validate_upstream_v5(c):
    """상위 파트는 선택 사항. 입력된 종목명/날짜/비교기업의 정합성은 검증한다."""
    if not (str(STOCK_CODE).isdigit() and len(str(STOCK_CODE))==6):
        raise ValueError('6자리 국내 종목코드가 필요합니다')
    import unicodedata
    norm=lambda s: re.sub(r"[^가-힣a-z0-9]", "", unicodedata.normalize("NFKC",str(s)).lower())
    if str(UPSTREAM_COMPANY_NAME).strip() and norm(UPSTREAM_COMPANY_NAME)!=norm(c['종목명']):
        raise ValueError(f'상위분석 대상 기업 불일치: {UPSTREAM_COMPANY_NAME} vs DART {c["종목명"]}')
    if str(UPSTREAM_ANALYSIS_ASOF).strip():
        ref=date.fromisoformat(UPSTREAM_ANALYSIS_ASOF)
        if ref>ASOF_DATE:raise ValueError('상위 분석 기준일이 미래입니다')
    peers=[str(x).zfill(6) for x in PEER_CODES]
    if len(set(peers))!=len(peers) or STOCK_CODE in peers:
        raise ValueError('비교기업 중복 또는 대상 기업 포함')
    if peers and len(peers)<2:
        raise ValueError('비교기업을 설정하는 경우 2개 이상 입력하세요(미설정은 빈 리스트)')
    return True


def latest_close_v5(code, asof=None):
    import FinanceDataReader as fdr
    d=asof or ASOF_DATE
    df=fdr.DataReader(code,(d-timedelta(days=40)).isoformat(),(d+timedelta(days=1)).isoformat())
    if df is None or df.empty: raise CollectError(f"{code}: 평가기준 종가 조회 실패")
    df=df[df.index.date<=d]
    if df.empty: raise CollectError(f"{code}: 기준일 이전 종가가 없습니다")
    dt=df.index[-1].date()
    if (d-dt).days>MAX_PRICE_AGE_DAYS:
        raise CollectError(f"{code}: 주가가 너무 오래됨 (마지막 {dt}, 평가 {d})")
    price=float(df.iloc[-1]["Close"])
    if not math.isfinite(price) or price<=0: raise CollectError("유효하지 않은 종가")
    return price,dt.isoformat()


def dart_published_filing_v5(corp_code,year,reprt_code,api_key,asof=None):
    """공시 목록을 먼저 확인. 제출일이 평가일 뒤라면 사용하지 않음."""
    d=asof or ASOF_DATE
    report_terms={"11013":"분기보고서", "11012":"반기보고서", "11014":"분기보고서", "11011":"사업보고서"}
    month={"11013":3,"11012":6,"11014":9,"11011":12}[reprt_code]
    bgn=date(year,month,1).isoformat().replace('-','')
    # 공시는 결산 이후 수주~수개월 뒤에 나올 수 있으므로 충분한 범위 조회
    res=http_get(f"{DART_URL}/list.json",{"crtfc_key":api_key,"corp_code":corp_code,
        "bgn_de":f"{year}0101","end_de":d.strftime('%Y%m%d'),"page_count":100}).json()
    if res.get("status")!="000":return None
    filings=[]
    for r in res.get("list",[]):
        nm=str(r.get("report_nm","")); rpt_dt=str(r.get("rcept_dt",""))
        if len(rpt_dt)!=8 or rpt_dt>d.strftime('%Y%m%d'):continue
        if report_terms[reprt_code] not in nm:continue
        if f"({year}.{month:02d})" not in nm and f"({year}.{month})" not in nm:continue
        filings.append(r)
    if not filings:return None
    # 최근 접수(정정 포함)의 공시를 추적한다.
    r=sorted(filings,key=lambda x:(x.get("rcept_dt",""),x.get("rcept_no","")))[-1]
    return {"rcept_no":r["rcept_no"],"rcept_dt":r["rcept_dt"],"report_nm":r["report_nm"]}


def interim_ttm_v5(stock_code, annual, year, api_key, asof=None, desired=None):
    """연차 + 당해/전년 동기 누적 매출·영업이익·지배순이익으로 TTM 구성.
    필수 항목이 누락되면 'TTM 추정'을 만들지 않고 연차 기준을 반환.
    """
    d=asof or ASOF_DATE
    c=copy.deepcopy(annual)
    c['valuation_period']="FY"; c['valuation_period_end']=f"{year}-12-31"
    c['valuation_filing_dt']=str(annual.get('rcept_no') or '')[:8]
    c['annual_core']=copy.deepcopy(annual)
    if desired=="FY":return c
    corp_code,_,_= _dart_all(stock_code,year,api_key)
    candidates=[("Q3","11014",9),("H1","11012",6),("Q1","11013",3)]
    if desired and desired not in {"Q3","H1","Q1"}:return None
    if desired:candidates=[x for x in candidates if x[0]==desired]
    for period,rc,month in candidates:
        try:
            filing=dart_published_filing_v5(corp_code,year+1,rc,api_key,d)
            if not filing:continue
            body=http_get(f"{DART_URL}/fnlttSinglAcntAll.json",{
                "crtfc_key":api_key,"corp_code":corp_code,"bsns_year":str(year+1),
                "reprt_code":rc,"fs_div":"CFS"}).json()
            if body.get("status")!="000":continue
            rows=body.get('list',[])
            # DART 당기·전기 동기 누계 비교. 해당 계정 ID만 허용.
            ids={"revenue":("ifrs-full_Revenue",("IS","CIS")),
                 "op_income":("dart_OperatingIncomeLoss",("IS","CIS")),
                 "ni_owner":("ifrs-full_ProfitLossAttributableToOwnersOfParent",("IS","CIS")),
                 "eq_owner":("ifrs-full_EquityAttributableToOwnersOfParent",("BS",)),
                 "cash":("ifrs-full_CashAndCashEquivalents",("BS",))}
            matched={}
            for k,(acct,sections) in ids.items():
                hits=[r for r in rows if r.get('account_id')==acct and r.get('sj_div') in sections
                      and (not r.get('account_detail') or r.get('account_detail')=='-')]
                if len(hits)!=1:continue
                r=hits[0]
                curr=_num(r.get('thstrm_amount'))
                prev=_num(r.get('frmtrm_amount'))
                if k in ('revenue','op_income','ni_owner'):
                    if curr is not None and prev is not None:matched[k]=(curr,prev)
                elif curr is not None:matched[k]=curr
            if not all(k in matched for k in ('revenue','op_income','ni_owner')):continue
            out=copy.deepcopy(c)
            for k in ('revenue','op_income','ni_owner'):
                if annual.get(k) is None:raise ValueError(f'{k}: 전기 연간 누락')
                current,previous=matched[k]
                out[k]=annual[k]+current-previous
                out['quality'][k]='exact'
                out.setdefault('ttm_lineage',{})[k]={
                    'annual_year':year,'annual_value':annual[k],
                    'current_ytd':current,'prior_ytd':previous,
                    'formula':'FY + current YTD - prior YTD',
                    'interim_rcept_no':filing['rcept_no'], 'rcept_dt':filing['rcept_dt'],
                    'source_url':f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={filing['rcept_no']}"}
            # 최신 공시 주식수 조회, 확인 안 되면 TTM 사용 거부 (연말 주식수와 섞지 않음)
            sh=http_get(f"{DART_URL}/stockTotqySttus.json",{
                "crtfc_key":api_key,"corp_code":corp_code,"bsns_year":str(year+1),"reprt_code":rc}).json()
            shares=None
            if sh.get('status')=='000':
                h=[r for r in sh.get('list',[]) if str(r.get('se','')).strip().startswith('보통주')]
                if len(h)==1:shares=_num(h[0].get('distb_stock_co')) or _num(h[0].get('istc_totqy'))
            if not shares or shares<=0:continue
            out['shares']=shares;out['quality']['shares']='exact'
            for k in ('eq_owner','cash'):
                if k in matched:out[k]=matched[k];out['quality'][k]='exact'
                else:out[k]=None;out['quality'][k]='missing'
            # 연간의 재투자·부채 검증값을 반기·분기 기준으로 자동 재사용하면 안 됨
            for k in ('da','capex','debt','cfo'):
                out[k]=None;out['quality'][k]='missing'
            out.update({'valuation_period':period,'valuation_period_end':f'{year+1}-{month:02d}-{calendar.monthrange(year+1,month)[1]}',
                        'valuation_filing_dt':filing['rcept_dt'], 'valuation_filing':filing['rcept_no'],
                        'valuation_report':filing['report_nm']})
            return out
        except Exception as e:
            print(f"⚠ {stock_code} {period} TTM 불채택: {type(e).__name__}: {str(e)[:130]}")
    if desired:return None
    return c


def apply_ttm_overrides_v5(c):
    """TTM 재무제표 원문과 기간이 맞는 승인치만 사용. 연차 승인치는 재사용 금지."""
    if c.get('valuation_period')=='FY':
        if VERIFIED_TTM_OVERRIDES:
            raise ValueError('TTM 보고서가 채택되지 않아 TTM 수동 검증값을 사용할 수 없음')
        return c
    for k, entry in VERIFIED_TTM_OVERRIDES.items():
        if k not in ('da','capex','debt') or not isinstance(entry,dict):
            raise ValueError(f'{k}: 허용된 검증 계정과 출처 구조 필요')
        required=('value','source_url','basis','asof','account_note')
        if any(field not in entry or entry[field] in (None,'') for field in required):
            raise ValueError(f'{k}: 검증 근거 필수 {required}')
        if entry['basis']!='연결' or entry['asof']!=c['valuation_period_end']:
            raise ValueError(f'{k}: 연결 기준 및 TTM 평가기간 종료일 일치 필요')
        if not str(entry['source_url']).startswith('https://dart.fss.or.kr/'):
            raise ValueError(f'{k}: DART 원문 링크 필요')
        if len(str(entry['account_note']).strip())<15:
            raise ValueError(f'{k}: 중복·합산 근거 설명 부족')
        v=float(entry['value'])
        if not math.isfinite(v) or v<0:
            raise ValueError(f'{k}: 음수/비유한 입력')
        c[k]=v;c['quality'][k]='reviewed'
        c.setdefault('ttm_lineage',{})[k]={
            'source_url':entry['source_url'],'value':v,'asof':entry['asof'],
            'account_note':entry['account_note'],'formula':'공시 원문 수동 검증 확정'}
    return c


def validate_current_basis_v5(c, price_tuple):
    if price_tuple is None or not price_tuple[0] or not c.get('shares'):
        raise ValueError('현재 종가 또는 공시 주식수 누락')
    dt=date.fromisoformat(price_tuple[1])
    if dt>ASOF_DATE or (ASOF_DATE-dt).days>MAX_PRICE_AGE_DAYS:
        raise ValueError('주가의 미래/과도한 지연')
    filing=c.get('valuation_filing_dt','')
    if filing and filing.isdigit() and len(filing)==8 and filing>dt.strftime('%Y%m%d'):
        raise ValueError(f'가격일 이후에 공개된 공시 사용 금지: {filing} > {dt}')
    return float(price_tuple[0]),dt.isoformat()

print('✅ v5 현재시점·TTM·상위 분석 인계 도구 준비 완료')

# v5: 원천 행의 기간/공시일 보정. 연차 API 원본과 TTM 파생값을 구분합니다.
EXTRA_RECORDS_V4 = extra_records_v2
def extra_records_v2(core_by_year,price_latest,stock_code):
    # 기존 추가 원천의 연차 행만 유지. TTM 파생값은 별도 출처 표에 기록.
    annual={y:(c.get('annual_core',c) if c.get('valuation_period')!='FY' else c)
            for y,c in core_by_year.items()}
    return EXTRA_RECORDS_V4(annual,price_latest,stock_code)


# v6: 공통엔진 15열 표 연결(열 이름/단위/연결/시점/공시 선행성 대조)
COMMON_ACCEPTED = {"매출액(연간)":"revenue", "영업이익(연간)":"op_income",
                   "지배주주순이익(연간)":"ni_owner", "지배자본총계":"eq_owner",
                   "현금및현금성자산":"cash"}
# 공통 w02 기본표의 당기순이익은 지배주주순이익과 다르므로 일부러 매핑하지 않는다.
def load_common_v6(path, stock, asof):
    """공통표 CSV 또는 DataFrame 수신. 외부 스키마는 명시적 column_map 필요."""
    if path is None or (isinstance(path,str) and not path.strip()):
        return None
    if isinstance(path,pd.DataFrame):
        df=path.copy()
    elif isinstance(path,(str,Path)):
        fp=Path(path)
        if not fp.is_file():raise ValueError(f'공통표 파일을 찾을 수 없습니다: {fp}')
        df=pd.read_csv(fp,encoding='utf-8-sig',dtype=str)
    else:
        raise TypeError('common_records에는 CSV 경로 또는 pandas.DataFrame만 전달하세요')
    mapping=globals().get('COMMON_COLUMN_MAP',{})
    if not isinstance(mapping,dict):raise ValueError('column_map은 {외부열: 표준열} 딕셔너리여야 합니다')
    unknown_src=[x for x in mapping if x not in df.columns]
    unknown_dest=[x for x in mapping.values() if x not in COLUMNS]
    if unknown_src or unknown_dest:
        raise ValueError(f'column_map 오류: 없는 원본열={unknown_src}, 지원하지 않는 대상열={unknown_dest}')
    if len(set(mapping.values()))!=len(mapping):
        raise ValueError('column_map의 대상 표준열 이름이 중복되었습니다')
    df=df.rename(columns=mapping)
    if len(df.columns)!=15 or set(df.columns)!=set(COLUMNS) or df.columns.duplicated().any():
        raise ValueError(f'공통표 열 형식 불일치. 필수 15열={COLUMNS}, 실제={list(df.columns)}. 필요하면 column_map을 명시하세요')
    df=df[COLUMNS].copy()
    df['종목코드']=df['종목코드'].astype(str).str.replace(r'\.0$','',regex=True).str.zfill(6)
    if not check_records(df,show=False):raise ValueError('공통표 형식 검증 실패(단위/날짜/상태 등 확인)')
    df=df.loc[df['종목코드']==stock].copy()
    if df.empty:raise ValueError(f'공통표에 종목 {stock}이 없습니다')
    return df

def common_annual_v6(df, stock, y, asof):
    """공식 연결·연간·원화·확인완료·시점 타당 행만 식별. 충돌과 중복은 실패."""
    if df is None:return {},{}
    selected,trace={},{}
    day=f'{y}-12-31'
    for _,r in df.iterrows():
        label=str(r['항목']).strip()
        if label not in COMMON_ACCEPTED:continue
        if str(r['기준']).strip()!='연결' or str(r['단위']).strip()!='원':continue
        if str(r['기준일'])[:10]!=day:continue
        if str(r['상태'])!=PASSED:continue
        disclosed=str(r['공시일'])[:10]
        if disclosed in ('','nan','None','해당 없음'):continue
        try:dt=date.fromisoformat(disclosed)
        except ValueError:continue
        if dt>asof or dt<date(y,12,31):continue
        # 공식 원천·출처URL 확인 (단순 형식통과와 실제 정확성은 별개)
        if not ('DART' in str(r['출처']).upper() or 'dart.fss.or.kr' in str(r['출처 URL']).lower()):continue
        k=COMMON_ACCEPTED[label]
        try:v=float(r['값'])
        except (TypeError,ValueError):continue
        if not math.isfinite(v):continue
        if k in selected:
            if not math.isclose(v,selected[k],rel_tol=1e-10,abs_tol=1):
                raise ValueError(f'공통표에서 {y} {k}가 서로 충돌합니다')
            # 동일 계정 중복은 출처 시점의 차이를 확인해야 하므로 차단
            raise ValueError(f'공통표에서 {y} {k} 행이 중복됐습니다. 최신 확정 한 행만 남겨주세요')
        selected[k]=v
        trace[k]={'출처':r['출처'],'공시일':disclosed,'출처 URL':r['출처 URL'], '값':v}
    return selected,trace

def reconcile_common_v6(annual, df, year, asof):
    """공통표 값이 있으면 DART 재조회 값과 대조한 후 공통 엔진 수치를 채택한다."""
    selected,trace=common_annual_v6(df,STOCK_CODE,year,asof)
    for k,v in selected.items():
        actual=annual.get(k)
        if actual is None:
            raise ValueError(f'공통표 {year} {k}은 있으나 DART 표준 원천에서 검증 불가')
        if not math.isclose(float(actual),v,rel_tol=1e-8,abs_tol=1):
            raise ValueError(f'공통표 ↔ DART {year} {k} 불일치: 공통표={v}, DART={actual}')
        annual[k]=v
        annual.setdefault('quality',{})[k]='exact'
    annual['common_trace']=trace
    return annual
print('✅ v6 공통표 15열 파서/실제 원천 교차검증 준비 완료')


✅ v2 밸류에이션 도구 준비 완료
✅ v5 현재시점·TTM·상위 분석 인계 도구 준비 완료
✅ v6 공통표 15열 파서/실제 원천 교차검증 준비 완료


## 2. DART 원천 + 최신 종가 + 공개된 최신 TTM 입력

분석 대상과 동종기업 모두 현재 종가를 조회하며, 비교기업은 **분석 대상과 같은 실적 기간(FY/Q1/H1/Q3)** 만 사용합니다. 해당 시점의 재무 공시가 공개됐다는 증거가 없으면 미사용합니다. TTM은 연차+당기 누적−전년 동기 누계 방식이며, 원본 연차·최신분기를 분리 보관합니다.


In [8]:
# 현재 평가 기준은 런타임 시점 (종가는 실시간 체결가가 아닌 최신 확인 가능한 종가)
DART_API_KEY = get_key("DART_API_KEY")
ASOF_DATE=datetime.now(ZoneInfo('Asia/Seoul')).date()
COMMON_TABLE=None
try:
    COMMON_TABLE=load_common_v6(COMMON_CSV_PATH,STOCK_CODE,ASOF_DATE)
    if COMMON_TABLE is not None:print(f'✅ 공통엔진 15열 표 수신: 대상 {STOCK_CODE} / {len(COMMON_TABLE)}행')
    elif COMMON_ONLY:raise ValueError('COMMON_ONLY=True인데 공통표 경로가 비었습니다')
    else:print('ℹ 공통표가 없어 DART 독립 실행 모드로 진행')
except Exception as e:
    raise ValueError(f'공통엔진 인계 실패(임의 대체 금지): {e}') from e
dart_years=[]
for year in (YEAR-1,YEAR):
    r=run_step(f"DART {year} 연차 재무",1,lambda y=year:collect_dart(STOCK_CODE,y,4,"CFS",DART_API_KEY))
    if r is not None:dart_years.append(r)
# 공통표가 있으면 해당 표를 기본 출처/지표 테이블로 사용하고,
# DART 별도 수집분은 핵심 계정·TTM·주식수 등 필수 보강에 활용한다.
dart=pd.concat(dart_years,ignore_index=True) if dart_years else None
if COMMON_TABLE is not None:
    dart=COMMON_TABLE.copy()
    print('✅ 출처·지표 공통표: 입력받은 15열 사용 (보강 수집 DART는 별도 원천 확인)')
core={};annual_core={};price_ye={};price_now=None;VALUATION_PRICE=None;SOURCE_TIMING=[]
if dart is None:
    print('⛔ DART 기본 재무를 가져오지 못했습니다')
else:
    for y in (YEAR-1,YEAR):
        try:
            c=dart_val_core(STOCK_CODE,y,DART_API_KEY)
            c=reconcile_common_v6(c,COMMON_TABLE,y,ASOF_DATE)
            annual_core[y]=c;core[y]=c
        except Exception as e:print(f'⚠ {y} 연차 핵심 계정 조회 실패:',str(e)[:200])
    if YEAR in core:
        try:
            validate_upstream_v5(core[YEAR])
            price_now=latest_close_v5(STOCK_CODE)
            # 가격 기준일까지 공시된 연차 보고서가 확인되는 경우에만 채택
            asof_price=date.fromisoformat(price_now[1])
            corp_code,_=dart_corp(STOCK_CODE,DART_API_KEY)
            annual_filing=dart_published_filing_v5(corp_code,YEAR,'11011',DART_API_KEY,asof_price)
            if annual_filing is None:
                raise ValueError(f'{YEAR} 연차사업보고서가 평가기준일 이전에 공시되었는지 확인 불가')
            core[YEAR]['rcept_no']=annual_filing['rcept_no']
            latest=interim_ttm_v5(STOCK_CODE,core[YEAR],YEAR,DART_API_KEY,asof_price)
            latest=apply_ttm_overrides_v5(latest)
            core[YEAR]=latest
            VALUATION_PRICE=validate_current_basis_v5(latest,price_now)
            SOURCE_TIMING.append({'대상':'분석기업','주가일':VALUATION_PRICE[1],
                '실적기간':latest['valuation_period'],'실적종료':latest['valuation_period_end'],
                '공시접수일':latest['valuation_filing_dt'],
                '주식수기준':'최종 채택 공시의 보통주 유통주식수'})
            if latest['valuation_period']=='FY':
                print('⚠ 최신 공개 분기 TTM 미채택: 연차 실적을 사용합니다(주가만 현재 기준).')
            if latest['valuation_period']!='FY' and VERIFIED_FINANCIAL_OVERRIDES:
                print('⚠ 연간 수동 확인 계정은 TTM에서 재사용하지 않음. DCF/EV·EBITDA는 추가 증빙 필요')
        except Exception as e:
            print('⛔ 현재 시점 기준 공시·주가·선택 인계 검증 실패:',type(e).__name__,str(e)[:250])
            VALUATION_PRICE=None
    if core and VALUATION_PRICE is not None:
        extra=extra_records_v2(core,VALUATION_PRICE,STOCK_CODE)
        if extra is not None:dart=pd.concat([dart,extra],ignore_index=True)
    print('평가 종가(현재 확인 가능한 마지막 거래일):',VALUATION_PRICE)
    if YEAR in core:print('실적기간:',core[YEAR].get('valuation_period'), '공시:',core[YEAR].get('valuation_filing_dt'))


ℹ 공통표가 없어 DART 독립 실행 모드로 진행
✅ DART 2024 연차 재무: 6줄 (확인 필요 0줄)
✅ DART 2025 연차 재무: 6줄 (확인 필요 0줄)
⚠ 최신 공개 분기 TTM 미채택: 연차 실적을 사용합니다(주가만 현재 기준).
평가 종가(현재 확인 가능한 마지막 거래일): (367500.0, '2026-11-13')
실적기간: FY 공시: 20260311


## 3. 출처·지표·모형 코드 사전점검

자동 집계 계정의 불확실성, 표준 계정 ID 불일치, 연결/별도 혼용은 그대로 `보류`합니다. 대상 기업의 최신 공시자료가 없다면 이전 연차 기준임을 명시합니다. 최신 주가와 주식수의 정확한 동일 날짜 보장은 공시 간 거래(증자·소각·분할)에 따라 별도 검토합니다.


In [9]:
dart_sources=dart_tagged=metrics=None
PRE=None;PROCEED=False
if dart is None or YEAR not in core or VALUATION_PRICE is None:
    print("⛔ DART/평가기준 주가/주식수 부족")
else:
    dart_sources,dart_tagged=build_sources(with_ids(dart))
    METRIC_SPECS_V = METRIC_SPECS + (
        MetricSpec("지배주주 순이익률","%","지배주주순이익(연간) ÷ 매출액(연간) × 100","ratio",("지배주주순이익(연간)","매출액(연간)")),
        MetricSpec("ROE(기말 지배자본)","%","지배주주순이익(연간) ÷ 지배자본총계 × 100","ratio",("지배주주순이익(연간)","지배자본총계")),
    )
    metrics=compute_metrics(dart_tagged,METRIC_SPECS_V,STOCK_CODE,f"{YEAR}-12-31")  # 연차 재무 수치만 표시하는 지표(현재 TTM 지표와 구분)
    PRE=prescreen_v2(dart_tagged,core,YEAR)
    PRE['facts'].update({'현재평가기준일':VALUATION_PRICE[1],
        '기초실적기간':core[YEAR].get('valuation_period'),
        '기초실적종료':core[YEAR].get('valuation_period_end'),
        '상위분석기준일':UPSTREAM_ANALYSIS_ASOF})
    # DART 누락값 또는 계정명 불명확 시 모형 자동 보류. 수동 검증값은 조건부까지만 허용.
    for model,required in {"PER":("ni_owner","shares"),"PBR":("eq_owner","shares"),
                            "EV/EBITDA":("op_income","da","debt","cash","shares"),
                            "DCF":("revenue","op_income","da","capex","debt","cash","shares"),
                            "PSR":("revenue","shares")}.items():
        unavailable=[k for k in required if core[YEAR].get(k) is None]
        if unavailable:
            PRE["model_caps"][model]=_status("보류","필수 계정 미확정: "+", ".join(unavailable))
        elif any(core[YEAR]["quality"].get(k)=="reviewed" for k in required):
            if PRE["model_caps"][model]["ceiling"]=="적합":PRE["model_caps"][model]["ceiling"]="조건부"
    display(pd.DataFrame([{"모형":m,**PRE["model_caps"][m]} for m in MODEL_ORDER]))
    if PRE["is_financial"]:print("⛔ 금융업 분석 범위 밖")
    elif all(v["ceiling"]=="보류" for v in PRE["model_caps"].values()): print("⛔ 모든 모형 보류")
    else:PROCEED=True


,모형,ceiling,reason,issues
0,PER,적합,양의 EPS,[]
1,PBR,적합,양의 지배자본과 BPS 확보,[]
2,EV/EBITDA,보류,"필수 계정 미확정: da, debt",[]
3,DCF,보류,"필수 계정 미확정: da, capex, debt",[]
4,PSR,적합,양의 매출과 주식수 확보,[]


## 4. 현재·과거·동종 배수

현재 배수 = **최신 종가 × 최신 공시 주식수 ÷ 실제 공시된 TTM(또는 FY) 실적**. 동종기업은 반드시 분석기업과 같은 실적기간을 채택합니다. 과거 PER은 사후 계산 지표로만 표시됩니다. 동종기업 최소 2개 유효 표본이 없으면 상대가치 모형을 보류합니다.


In [10]:
HIST5=PEERS=CUR=None;BANDS={};PEER_SOURCE_TIMING=[]
if not PROCEED:print('⏭ 건너뜀')
else:
    CUR=multiples_from_core(core[YEAR],VALUATION_PRICE[0])
    CUR.update({'기준일':VALUATION_PRICE[1], '평가기준주가':VALUATION_PRICE[0],
        '실적기간':core[YEAR].get('valuation_period'),'실적종료':core[YEAR].get('valuation_period_end')})
    hist=[]
    for y in range(YEAR-4,YEAR+1):
        try:
            c=dart_val_core(STOCK_CODE,y,DART_API_KEY)
            ye,_=krx_price(STOCK_CODE,y);p,dt=validate_market_basis(c,ye,y)
            mm=multiples_from_core(c,p)
            hist.append({'연도':y,'연말거래일':dt,'연말주가':p,'PER':mm['PER'],'PBR':mm['PBR'],
                         'EV/EBITDA':mm['EV/EBITDA'],'PSR':mm['PSR'],
                         '해석':'사후 연말가격+확정연차실적; 당시 관측 PER 아님'})
        except Exception as e:print(f'⚠ {y} 역사 배수 제외:',str(e)[:130])
    HIST5=pd.DataFrame(hist)
    peers=[]; period=core[YEAR].get('valuation_period','FY')
    eval_date=date.fromisoformat(VALUATION_PRICE[1])
    for code_ in PEER_CODES:
        try:
            annual=dart_val_core(code_,YEAR,DART_API_KEY)
            c=interim_ttm_v5(code_,annual,YEAR,DART_API_KEY,eval_date,desired=period)
            if c is None:raise ValueError(f'{period} 동일 기간 공시 없음')
            p,dt=latest_close_v5(code_,eval_date)
            if dt!=VALUATION_PRICE[1]:raise ValueError('비교기업 종가 기준일 불일치')
            price=validate_current_basis_v5(c,(p,dt))
            mm=multiples_from_core(c,price[0])
            peers.append({'종목코드':code_,'종목명':c['종목명'],'현재거래일':dt,
                '실적기간':c.get('valuation_period'), '공시접수일':c.get('valuation_filing_dt'),
                'PER':mm['PER'],'PBR':mm['PBR'],'EV/EBITDA':mm['EV/EBITDA'],'PSR':mm['PSR']})
            PEER_SOURCE_TIMING.append({'종목코드':code_,'종목명':c['종목명'],'주가일':dt,
                '실적기간':c.get('valuation_period'),'실적종료':c.get('valuation_period_end'),
                '공시접수일':c.get('valuation_filing_dt')})
        except Exception as e:print(f'⚠ 동종기업 {code_} 제외:',str(e)[:180])
    PEERS=pd.DataFrame(peers)
    BANDS=peer_reference_bands(PEERS)
    MULTIPLE_REFERENCE_SOURCE={}
    for m in ('PER','PBR','EV/EBITDA','PSR'):
        if m in BANDS:
            MULTIPLE_REFERENCE_SOURCE[m]='동종기업 동기간 중앙값'
        elif HIST5 is not None and not HIST5.empty and m in HIST5.columns:
            vals=pd.to_numeric(HIST5[m],errors='coerce')
            vals=vals[(vals>0)&np.isfinite(vals)]
            if len(vals)>=3:
                med=float(vals.median())
                BANDS[m]=(PEER_MULT_LOWER*med,PEER_MULT_UPPER*med,med)
                MULTIPLE_REFERENCE_SOURCE[m]='자기 회사 과거 연말 확정연차 사후 배수 중앙값(조건부)'
        if m not in BANDS:
            PRE['model_caps'][m]=_status('보류','동종기업 2개 또는 자기 회사 과거 유효배수 3개 미만')
        elif PRE['model_caps'][m]['ceiling']=='적합':
            PRE['model_caps'][m]['ceiling']='조건부'
    print('배수 기준 출처:',MULTIPLE_REFERENCE_SOURCE)
    display(pd.DataFrame([{'모형':m,**PRE['model_caps'][m]} for m in MODEL_ORDER]))
    display(pd.DataFrame([CUR])); display(HIST5); display(PEERS)
    print('동종기업 배수 허용 범위:',BANDS)
    if all(x['ceiling']=='보류' for x in PRE['model_caps'].values()):PROCEED=False


배수 기준 출처: {'PER': '자기 회사 과거 연말 확정연차 사후 배수 중앙값(조건부)', 'PBR': '자기 회사 과거 연말 확정연차 사후 배수 중앙값(조건부)', 'PSR': '자기 회사 과거 연말 확정연차 사후 배수 중앙값(조건부)'}


,모형,ceiling,reason,issues
0,PER,조건부,양의 EPS,[]
1,PBR,조건부,양의 지배자본과 BPS 확보,[]
2,EV/EBITDA,보류,동종기업 2개 또는 자기 회사 과거 유효배수 3개 미만,[]
3,DCF,보류,"필수 계정 미확정: da, capex, debt",[]
4,PSR,조건부,양의 매출과 주식수 확보,[]


,PER,PBR,EV/EBITDA,PSR,시가총액,EV,EBITDA,순차입금,기준일,평가기준주가,실적기간,실적종료
0,12.602284,0.757431,None,0.610094,2.144064e+12,None,None,None,2026-11-13,367500.0,FY,2025-12-31


,연도,연말거래일,연말주가,PER,PBR,EV/EBITDA,PSR,해석
0,2021,2021-12-30,318500.0,18.449390,0.843087,None,0.691618,사후 연말가격+확정연차실적; 당시 관측 PER 아님
1,2022,2022-12-29,357000.0,17.777811,0.895940,None,0.659749,사후 연말가격+확정연차실적; 당시 관측 PER 아님
2,2023,2023-12-28,407000.0,13.690896,0.968439,None,0.690071,사후 연말가격+확정연차실적; 당시 관측 PER 아님
3,2024,2024-12-30,374000.0,13.748760,0.809389,None,0.628924,사후 연말가격+확정연차실적; 당시 관측 PER 아님
4,2025,2025-12-30,432500.0,14.831260,0.891399,None,0.718002,사후 연말가격+확정연차실적; 당시 관측 PER 아님


""


동종기업 배수 허용 범위: {'PER': (7.41562982298557, 29.66251929194228, 14.83125964597114), 'PBR': (0.4456993564176095, 1.782797425670438, 0.891398712835219), 'PSR': (0.3450354190567109, 1.3801416762268437, 0.6900708381134218)}


## 5. 모형 적합성 판정 + 주력/보조 모형 선정 (Gemini)

Gemini가 하는 일은 두 가지뿐입니다.

1. 코드 상한 안에서 `적합 → 조건부 → 보류` 방향으로만 낮추기
2. 최종 `보류`가 아닌 모형 중 주력·보조 모형 선택

코드는 AI가 상한을 넘겼는지와 보류 모형을 선택했는지 다시 검사합니다.


In [11]:
DATA_TABLE=dart_tagged[["rid","항목","값","단위","기준","기준일","공시일","출처","상태"]].to_csv(index=False) if dart_tagged is not None else ""
CTX=(f"종목: {core[YEAR]['종목명']} ({STOCK_CODE}) / "
     f"기업유형(상위 분석): {UPSTREAM_COMPANY_TYPE or '미제공'} / 산업(상위 분석): {UPSTREAM_INDUSTRY or '미제공'} / "
     f"상위분석기준일: {UPSTREAM_ANALYSIS_ASOF or '미제공'} / "
     f"현재 주가 기준일: {VALUATION_PRICE[1] if VALUATION_PRICE else '미정'} / "
     f"채택 실적: {core[YEAR].get('valuation_period')}({core[YEAR].get('valuation_period_end')}) / "
     f"상위 산업·기업 분석 핵심: {UPSTREAM_ANALYSIS_BRIEF or '미제공(종목 공시와 수치만 사용; 산업 특수 주장 금지)'}"
     if YEAR in core else '')
P1=r"""
[역할] 밸류에이션 모형의 적합성을 판정하는 애널리스트.
[종목] {{CTX}}
[절대 규칙] 상위 정보가 없으면 산업·기업 유형을 추측하거나 새로운 사실로 채우지 않는다. 상위 정보가 있으면 재분류하지 않는다. 코드를 통한 상한을 높이면 안 된다. 보류 모형 선택 금지. 답은 JSON 하나.
[코드 상한] {{CAPS}}
[재무표] {{DATA_TABLE}}
[출력 JSON]
{"models":[{"model":"PER","status":"조건부","reason":"..."}],
 "primary_model":"PER","secondary_model":"없음","selection_reason":"...","data_issues":[]}
각 모델 5개 모두 models 배열에 유일하게 포함. 불확실한 모형은 보류.
"""
step1=MODEL_TABLE=None
STAGE1_OK=False
PROCEED=bool(PROCEED)
if PROCEED:
    prompt1=fill(P1,CTX=CTX,CAPS=json.dumps(PRE["model_caps"],ensure_ascii=False),DATA_TABLE=DATA_TABLE)
    try:step1,err=parse_json_obj(ask_json(prompt1))
    except Exception as e:step1,err=None,str(e)
    if err or not step1:print("⛔ 모형선정 실패:",err);PROCEED=False
    else:
        entries=step1.get("models",[])
        ids=[r.get("model") for r in entries if isinstance(r,dict)]
        if len(entries)!=5 or sorted(ids)!=sorted(MODEL_ORDER):
            print("⛔ 모형 판정 응답에 중복/누락 있음");PROCEED=False
        else:
            MODEL_TABLE=apply_ai_caps(PRE["model_caps"],entries)
            usable=MODEL_TABLE.loc[MODEL_TABLE["final_status"]!="보류","model"].tolist()
            pm=step1.get("primary_model");sm=step1.get("secondary_model")
            if pm not in usable or (sm!="없음" and (sm not in usable or sm==pm)):
                print("⛔ AI 주력·보조 선정이 유효하지 않음. 자동 대체하지 않습니다");PROCEED=False
            elif not usable:PROCEED=False
            else:
                STAGE1_OK=True
                display(MODEL_TABLE);print("주력:",pm,"보조:",sm)


⏳ Gemini 일시 오류 HTTP 504 (gemini-3.8-flash) · 1회 실패 · 다음 시도 2초 후 · 다음 모델 gemini-3.5-flash-lite · 총 경과 28초 (Colab ■로 중단 가능)
✅ Gemini 응답 완료: gemini-3.5-flash-lite (총 호출 시도 2회)


,model,code_ceiling,ai_status,final_status,code_reason,ai_reason
0,PER,조건부,조건부,조건부,양의 EPS,양의 EPS
1,PBR,조건부,조건부,조건부,양의 지배자본과 BPS 확보,양의 지배자본과 BPS 확보
2,EV/EBITDA,보류,보류,보류,동종기업 2개 또는 자기 회사 과거 유효배수 3개 미만,동종기업 2개 또는 자기 회사 과거 유효배수 3개 미만
3,DCF,보류,보류,보류,"필수 계정 미확정: da, capex, debt","필수 계정 미확정: da, capex, debt"
4,PSR,조건부,조건부,조건부,양의 매출과 주식수 확보,양의 매출과 주식수 확보


주력: PER 보조: PBR


## 6. Bear / Base / Bull 실적·가치평가 가정 (Gemini)

- **사용 모형에 필요한 필드만 필수**: 예를 들어 PER·PBR·PSR만 사용하면 `target_ev_ebitda`, `wacc`는 `null`이어도 됩니다.
- 보류 모형에 쓸 근거 없는 값을 만들어내지 않습니다.
- 실행 실패 후 **이 셀만 다시 눌러도**, 앞선 모형선정이 통과했다면 다시 API를 호출합니다. 하지만 데이터/모형선정 실패는 우회하지 않습니다.
- Gemini 혼잡은 5초/10초 재시도 + 예비 모델 1회로 제한합니다.


In [12]:
P2=r"""
[역할] 밸류에이션용 Bear/Base/Bull 가정만 작성. 모든 계산은 Python. 오직 JSON 객체 출력.
[종목] {{CTX}}
[최종 모형 상태] {{MODEL_TABLE}}
[사용 모형] {{USABLE}}
[시나리오별 필수 필드] {{REQUIRED}}
[현재/과거 배수] {{MULT}}
[비교기업 배수: 미제공일 수 있음] {{PEERS}}
[코드가 정한 배수 범위] {{BANDS}}
[기초 재무] {{FACTS}}
[사업부문 매출] {{SEG}}
[규칙]
- bear, base, bull 3개 모두 만들고, 필수 필드는 반드시 유한한 JSON 숫자.
- 비사용 모형 배수는 null로 출력한다. 숫자를 억지로 만들어내지 않는다.
- 사용 모형의 목표 배수는 코드가 제공한 허용범위 안에서만 입력한다. 비교기업이 없으면 자기기업의 과거 사후 배수라는 점을 가정 근거·한계에 반드시 명시한다.
- 비율은 소수로 입력(5%=0.05). 수치가 불명확하면 없는 것으로 처리하고 limitation에 기록.
- Bear <= Base <= Bull (매출/영업이익/순이익/주당 가치).
- DCF 미사용 시 wacc/terminal_growth/FCFF용 가정은 null 허용.
- DCF 사용 시 WACC-g >= 0.02, ΔNWC=0 가정.
- 성장률 근거는 assumption_notes에 item='growth'와 rationale 문장 필수.
- 제공되지 않은 외부 전망치·기업 고유 사실을 만들지 않는다. 상위 산업·기업분석의 정의를 변경하지 않는다.
- 현재 종가에 적용하는 1년 선행 실적 평가이며 과거 연말 가격으로 평가하지 않는다.
[출력 예시: JSON 한 개]
{"scenarios":{"bear":{"growth":0.0,"op_margin":0.0,"net_to_op":0.0},
              "base":{"growth":0.0,"op_margin":0.0,"net_to_op":0.0},
              "bull":{"growth":0.0,"op_margin":0.0,"net_to_op":0.0}},
 "assumption_notes":[{"item":"growth","source_type":"분석자 가정","rationale":"근거 설명"}],
 "limitation":[],"data_issues":[]}
"""
# 단계 재실행 시 이전 가정 검증 실패(PROCEED=False) 때문에 Gemini 호출이 건너뛰어지는 현상 방지.
# 단, 이전 모형 선정이 실패한 경우 절대 재개하지 않는다.
PROCEED=bool(globals().get("STAGE1_OK",False))
step2=None
SCENARIO_ERRORS=[]
if not PROCEED:
    print("⏭ 모형선정 단계가 통과하지 않아 가정 생성을 중단합니다. 앞 단계부터 확인하세요")
else:
    mult_txt=json.dumps(CUR,ensure_ascii=False,default=str)+"\n"+(HIST5.to_csv(index=False) if HIST5 is not None else "")
    required_keys=sorted(scenario_required_keys(usable))
    prompt2=fill(P2,CTX=CTX,MODEL_TABLE=MODEL_TABLE.to_csv(index=False),
                 USABLE=json.dumps(usable,ensure_ascii=False),
                 REQUIRED=json.dumps(required_keys,ensure_ascii=False),
                 MULT=mult_txt,
                 PEERS=PEERS.to_csv(index=False) if PEERS is not None else "없음",
                 FACTS=json.dumps(PRE["facts"],ensure_ascii=False,default=str),
                 SEG=json.dumps(SEGMENTS,ensure_ascii=False) if SEGMENTS else "없음",
                 BANDS=json.dumps(BANDS,ensure_ascii=False))
    try:
        raw=ask_json(prompt2)
        step2,err=parse_json_obj(raw)
    except Exception as e:
        step2,err=None,str(e)
    if err or not isinstance(step2,dict):
        SCENARIO_ERRORS=[f"Gemini 응답/JSON 파싱 실패: {err or '객체 아님'}"]
    else:
        step2=normalize_scenarios_v4(step2,usable)
        SCENARIO_ERRORS=validate_scenarios_v3(step2,core[YEAR],BANDS,usable,DCF_YEARS)
    if SCENARIO_ERRORS:
        PROCEED=False
        print("⛔ 시나리오 생성 또는 검증 실패. 재실행 강제하지 않습니다:")
        for issue in SCENARIO_ERRORS: print(" -",issue)
        if isinstance(step2,dict):
            print("참고: 실제 생성된 배수 (보류 모형의 null은 정상)")
            for label,values in step2.get("scenarios",{}).items():
                if isinstance(values,dict): print(label, {k:values.get(k) for k in MODEL_TARGET_KEYS.values()})
    else:
        print("✅ 가정 유효성 검증 통과. 사용 모형:",usable)
        display(pd.DataFrame(step2["scenarios"]).T)


⏳ Gemini 일시 오류 HTTP 504 (gemini-3.8-flash) · 1회 실패 · 다음 시도 2초 후 · 다음 모델 gemini-3.5-flash-lite · 총 경과 30초 (Colab ■로 중단 가능)
✅ Gemini 응답 완료: gemini-3.5-flash-lite (총 호출 시도 2회)
✅ 가정 유효성 검증 통과. 사용 모형: ['PER', 'PBR', 'PSR']


,growth,op_margin,net_to_op,payout,target_per,target_pbr,target_psr,tax_rate,da_to_revenue,capex_to_revenue,target_ev_ebitda
bear,-0.02,0.045,0.90,0.20,7.41563,0.445699,0.345035,0.0,0.0,0.0,NaN
base,0.02,0.052,0.92,0.25,14.83126,0.891399,0.690071,0.0,0.0,0.0,NaN
bull,0.05,0.060,0.95,0.30,22.00000,1.300000,1.000000,0.0,0.0,0.0,NaN


## 7. Python 시나리오별 가치평가 + 1년 선행 매출·이익 추정

재무 추정의 기초는 현재 이용할 수 있는 **TTM(혹은 최신 연간) 실적**입니다. 상대가치 모형은 다음 1년 선행값에 배수를 적용하고, DCF는 5년 추정을 사용합니다. 결산연도와 동일한 연도 예측으로 잘못 표시하지 않습니다.


In [13]:
CALC={}
FORECAST=None
if not PROCEED:
    print("⏭ 건너뜀")
else:
    c=core[YEAR]
    base={"revenue":c["revenue"],"equity":c["eq_owner"],"shares":c["shares"],
          "net_debt":(c["debt"]-c["cash"] if c.get("debt") is not None and c.get("cash") is not None else None),"segments":SEGMENTS}
    usable=MODEL_TABLE.loc[MODEL_TABLE["final_status"]!="보류","model"].tolist()
    rows=[]
    forecast_rows=[]
    for scen in ("bear","base","bull"):
        a=step2["scenarios"][scen]
        proj=project_financials(a,base,DCF_YEARS)
        y1=proj.iloc[0]
        forecast_rows.append({"시나리오":scen,"예상매출(억원)":y1["revenue"]/1e8,
                              "예상영업이익(억원)":y1["op"]/1e8,"예상순이익(억원)":y1["ni"]/1e8,
                              "예상EPS(원)":y1["ni"]/base["shares"],"영업이익률(%)":a["op_margin"]*100})
        for m in usable:
            try:
                r=fair_value_v2(m,a,base,DCF_YEARS)
                r.update({"모형":m,"시나리오":scen,"현재주가":VALUATION_PRICE[0],"현재주가기준일":VALUATION_PRICE[1],"실적기초기간":c.get("valuation_period"),
                          "괴리율(%)":(r["주당가치(원)"]/VALUATION_PRICE[0]-1)*100,
                          "극단값검사":model_extreme_flag(r["주당가치(원)"],VALUATION_PRICE[0])})
                rows.append(r)
            except Exception as e:
                print(f"⚠ {m}/{scen} 계산 실패:",e)
    CALC=pd.DataFrame(rows)
    FORECAST=pd.DataFrame(forecast_rows)
    if len(CALC)!=len(usable)*3:
        print("⛔ 계산 누락: 모형×3개 시나리오가 모두 필요");PROCEED=False
    else:
        display(FORECAST)
        display(CALC[["모형","시나리오","주당가치(원)","괴리율(%)","극단값검사"]])


,시나리오,예상매출(억원),예상영업이익(억원),예상순이익(억원),예상EPS(원),영업이익률(%)
0,bear,34440.291901,1549.813136,1394.831822,23907.899814,4.5
1,base,35846.018101,1863.992941,1714.873506,29393.524959,5.2
2,bull,36900.312751,2214.018765,2103.317827,36051.594957,6.0


,모형,시나리오,주당가치(원),괴리율(%),극단값검사
0,PER,bear,177292.134864,-51.757242,정상
1,PBR,bear,224774.588448,-38.836847,정상
2,PSR,bear,203680.795828,-44.576654,정상
3,PER,base,435943.000581,18.623946,정상
4,PBR,base,452151.012736,23.034289,정상
5,PSR,base,423988.595396,15.371046,정상
6,PER,bull,793135.089061,115.819072,정상
7,PBR,bull,663557.245196,80.559795,정상
8,PSR,bull,632484.122058,72.104523,정상


## 8. 확정 연차 실적·현재 TTM·과거 차트·AI 해석

회계상 확정 연차와 TTM 파생값은 분리 표기합니다. Gemini는 기업분석을 새로 쓰지 않고 앞 파트 정보가 있으면 이를 전제로 하고, 없으면 재무 수치·모형 비교만 현재 가치평가 결과와 필요한 확인사항만 해석합니다.


In [14]:
ACTUALS=None
CHART=None
AI_SUMMARY=""
if PROCEED:
    actual_rows=[]
    for y in (YEAR-1,YEAR):
        c=annual_core[y]
        actual_rows.append({"연도":y,"매출(억원)":c.get("revenue",np.nan)/1e8 if c.get("revenue") is not None else np.nan,
                            "영업이익(억원)":c.get("op_income",np.nan)/1e8 if c.get("op_income") is not None else np.nan,
                            "순이익(억원)":c.get("ni_owner",np.nan)/1e8 if c.get("ni_owner") is not None else np.nan,
                            "EPS(원)":c.get("ni_owner",np.nan)/c.get("shares",np.nan) if c.get("ni_owner") is not None and c.get("shares") else np.nan,
                            "영업이익률(%)":c.get("op_income",np.nan)/c.get("revenue",np.nan)*100 if c.get("op_income") is not None and c.get("revenue") else np.nan})
    ACTUALS=pd.DataFrame(actual_rows)
    if core[YEAR].get('valuation_period')!='FY':
        c=core[YEAR]
        ACTUALS=pd.concat([ACTUALS,pd.DataFrame([{'연도':f"TTM {c['valuation_period_end']}",
            '매출(억원)':c['revenue']/1e8,'영업이익(억원)':c['op_income']/1e8,
            '순이익(억원)':c['ni_owner']/1e8,'EPS(원)':c['ni_owner']/c['shares'],
            '영업이익률(%)':c['op_income']/c['revenue']*100}])],ignore_index=True)
    CHART=make_per_chart(HIST5,f"per_5y_{STOCK_CODE}.png") if HIST5 is not None else None

    P3=r"""
[역할] 대학 투자학회 밸류에이션 브리프의 해석 문단 작성.
[종목] {{CTX}}
[모형표] {{MODELS}}
[주력/보조] {{SELECT}}
[시나리오 가치] {{CALC}}
[가정 한계] {{LIMIT}}

[규칙]
- 표에 있는 숫자를 새로 계산하거나 바꾸지 않는다.
- 매수/매도/목표주가 권유 표현 금지.
- 3개 짧은 문단만: ①왜 주력/보조인지 ②모형 간 값 차이를 만드는 핵심 가정 ③검증관이 볼 항목.
- 확인 불가능한 기업 사실을 추가하지 않는다. 상위 분석 미제공 시 업종 특수 사실을 지어내지 않는다. 제공 시 그 분류를 바꾸지 않는다.
- 평가는 최신 종가와 최신 공시 TTM(또는 FY)에 근거하며, 투자의견은 작성하지 않는다.
"""
    prompt3=fill(P3,CTX=CTX,MODELS=MODEL_TABLE.to_csv(index=False),
                 SELECT=json.dumps({"primary":step1["primary_model"],"secondary":step1["secondary_model"],"reason":step1.get("selection_reason","")},ensure_ascii=False),
                 CALC=CALC[["모형","시나리오","주당가치(원)","괴리율(%)","극단값검사"]].to_csv(index=False),
                 LIMIT=json.dumps(step2.get("limitation",[]),ensure_ascii=False))
    try: AI_SUMMARY=ask_text(prompt3)
    except Exception as e: AI_SUMMARY=f"AI 해석 생성 실패: {e}"
    display(Markdown(AI_SUMMARY))


⏳ Gemini 일시 오류 HTTP 504 (gemini-3.8-flash) · 1회 실패 · 다음 시도 2초 후 · 다음 모델 gemini-3.5-flash-lite · 총 경과 29초 (Colab ■로 중단 가능)
✅ Gemini 응답 완료: gemini-3.5-flash-lite (총 호출 시도 2회)


농심(004370)의 밸류에이션 브리프 해석 문단은 다음과 같습니다.

본 브리프에서 주력 모형으로 PER, 보조 모형으로 PBR을 채택한 이유는 채택 실적(FY 2025-12-31) 기준 양의 지배주주순이익과 지배자본이 모두 확보되어 있어 수익성과 안정성 지표를 충실히 반영할 수 있기 때문입니다. EV/EBITDA 및 DCF 모형은 감가상각비, 자본지출, 이자부채 등 필수 계정의 미확정으로 보류되었으며, 동종기업 데이터 부재로 인해 자기기업 과거 사후 배수 범위 내에서 목표 배수를 설정하여 산술적 가치를 도출했습니다.

각 모형의 시나리오별 주당가치는 Bear, Base, Bull 각 국면에 따라 뚜렷한 괴리율을 보입니다. Base 시나리오 기준 PER은 435,943원(괴리율 18.62%), PBR은 452,151원(괴리율 23.03%), PSR은 423,988원(괴리율 15.37%)으로 유사한 수준의 가치를 형성하는 반면, Bear 시나리오에서는 PER 177,292원(-51.76%), PBR 224,774원(-38.84%) 등으로 하락 폭에 차이가 발생합니다. 이러한 값의 차이는 각 모형이 적용하는 수익, 자본, 매출 기준의 민감도와 배수 가정의 적용 방식에서 기인합니다.

검증관은 상위 산업 및 기업 분석 내용이 미제공된 상태에서 공시된 재무제표 수치만을 기반으로 수립된 가정의 타당성을 중점적으로 살펴보아야 합니다. 특히 EV/EBITDA와 DCF가 보류된 제약 속에서, 과거 사후 배수 범위를 바탕으로 설정된 목표 배수의 현실성과 극단값 검사에서 모두 정상 범위로 확인된 각 시나리오별 괴리율의 재무적 함의를 면밀히 검토해야 합니다.

## 9. 코드 검증

리포트 본문에는 검증 로그를 노출하지 않고 인계 폴더에 저장합니다.

검사:
- AI 판정 ≤ 코드 상한
- 주력/보조가 보류가 아님
- 모든 비보류 모형의 Bear/Base/Bull 계산 존재
- DCF의 WACC > g
- 재계산 일치
- 극단값(현재가 0.2배 미만 / 5배 초과) 경고
- DART 추출 품질과 확인 필요 줄 기록


In [15]:
LOG=[];HARD_FAILURES=[];WARNINGS=[]
def log(step,ok,msg,hard=True):
    LOG.append({"단계":step,"결과":"통과" if ok else ("차단" if hard else "확인 필요"),"내용":msg})
    if not ok:(HARD_FAILURES if hard else WARNINGS).append(f"{step}: {msg}")
    print(("✅ " if ok else "⛔ " if hard else "⚠ ")+msg)

if PROCEED:
    usable=MODEL_TABLE.loc[MODEL_TABLE["final_status"]!="보류","model"].tolist()
    log("모형상한",all(STATUS_RANK[r["final_status"]]<=STATUS_RANK[r["code_ceiling"]] for _,r in MODEL_TABLE.iterrows()),"AI 판정 상한 점검")
    log("주력/보조",step1["primary_model"] in usable and (step1["secondary_model"]=="없음" or step1["secondary_model"] in usable),"주력·보조 모두 비보류")
    log("시나리오완전성",len(CALC)==3*len(usable) and len(set(zip(CALC["모형"],CALC["시나리오"])))==len(CALC),"각 모형×3개 시나리오")
    errs=validate_scenarios_v3(step2,core[YEAR],BANDS,usable,DCF_YEARS)
    log("가정검증",not errs,"가정·시나리오 경제적 순서: "+("통과" if not errs else "; ".join(errs)))
    recalc=[]
    for _,r in CALC.iterrows():
        try:
            x=independent_fair_value_check(r["모형"],step2["scenarios"][r["시나리오"]],base,DCF_YEARS)
            if not math.isclose(x,float(r["주당가치(원)"]),rel_tol=1e-10,abs_tol=1e-6):
                recalc.append(r["모형"]+"/"+r["시나리오"])
        except Exception as e:recalc.append(str(e))
    log("독립검산",not recalc,"독립 수식 기반 가치 계산 " + ("모두 일치" if not recalc else ", ".join(recalc)))
    log('평가기준일',VALUATION_PRICE is not None and (ASOF_DATE-date.fromisoformat(VALUATION_PRICE[1])).days<=MAX_PRICE_AGE_DAYS,
        '최근 확인 가능한 종가, 7일 이내')
    log('공시선행성', not core[YEAR].get('valuation_filing_dt') or
        core[YEAR].get('valuation_filing_dt','')<=VALUATION_PRICE[1].replace('-',''),
        '채택 공시 접수일이 주가 날짜를 넘지 않음')
    log('동종기간동일', PEERS is not None and (PEERS.empty or
        (PEERS['실적기간']==core[YEAR].get('valuation_period')).all()), '동종기업과 실적기간 동일')
    try:
        up_ok=validate_upstream_v5(core[YEAR])
    except Exception: up_ok=False
    log('선택 상위분석',up_ok,'종목코드 및 선택 제공 기업명·공시 전제 일치(미제공 허용)')
    if COMMON_TABLE is None:
        log('공통표 교차대조', True, '독립 실행: 공통표 미입력으로 DART 원천 사용')
    else:
        log('공통표 교차대조', all('common_trace' in annual_core[y] for y in (YEAR-1,YEAR)),
            '공통표 제공 시 DART 실값과 대조한 기록 필요')
    log("원천필수",all(core[YEAR]["quality"].get(k) in ("exact","reviewed") for m in usable for k in
        {"PER":("ni_owner","shares"),"PBR":("eq_owner","shares"),
         "EV/EBITDA":("op_income","da","debt","cash","shares"),
         "DCF":("revenue","op_income","da","capex","debt","cash","shares"),
         "PSR":("revenue","shares")}[m]),"필수 원천은 표준계정 또는 근거 기록된 수동확정")
    extreme=CALC[CALC["극단값검사"]=="경고"]
    log("극단값",len(extreme)==0,f"극단가치 경고 {len(extreme)}건",hard=False)
    review_n=int((dart_tagged["상태"]=="확인 필요").sum())
    log("공통표",review_n==0,f"확인 필요 {review_n}줄(전체 테이블; 모형 필수 원천과 별개)",hard=False)
    if "DCF" in usable:
        SENSITIVITY=dcf_sensitivity(step2["scenarios"]["base"],base,DCF_YEARS)
        log("DCF민감도",SENSITIVITY["주당가치(원)"].notna().all(),"WACC ±1%p · g ±0.5%p 범위",hard=False)
    else:SENSITIVITY=pd.DataFrame()
    # 5년 전망이 1년과 일치하는지 검증
    log("추정일치",all(math.isclose(float(FORECAST.loc[FORECAST["시나리오"]==s,"예상매출(억원)"].iloc[0])*1e8,
        float(project_financials(step2["scenarios"][s],base,DCF_YEARS).iloc[0]["revenue"]),rel_tol=1e-10)
        for s in ("bear","base","bull")),"1년차 추정 매출 재검산")
    PROCEED=not HARD_FAILURES
    display(pd.DataFrame(LOG))
    if HARD_FAILURES:print("⛔ 핵심 검증 실패로 보고서를 생성하지 않습니다.")
else:
    log("실행상태",False,"필수 입력·모형·가정·계산 중 오류로 검증 불가")


✅ AI 판정 상한 점검
✅ 주력·보조 모두 비보류
✅ 각 모형×3개 시나리오
✅ 가정·시나리오 경제적 순서: 통과
✅ 독립 수식 기반 가치 계산 모두 일치
✅ 최근 확인 가능한 종가, 7일 이내
✅ 채택 공시 접수일이 주가 날짜를 넘지 않음
✅ 동종기업과 실적기간 동일
✅ 종목코드 및 선택 제공 기업명·공시 전제 일치(미제공 허용)
✅ 독립 실행: 공통표 미입력으로 DART 원천 사용
✅ 필수 원천은 표준계정 또는 근거 기록된 수동확정
✅ 극단가치 경고 0건
✅ 확인 필요 0줄(전체 테이블; 모형 필수 원천과 별개)
✅ 1년차 추정 매출 재검산


,단계,결과,내용
0,모형상한,통과,AI 판정 상한 점검
1,주력/보조,통과,주력·보조 모두 비보류
2,시나리오완전성,통과,각 모형×3개 시나리오
3,가정검증,통과,가정·시나리오 경제적 순서: 통과
4,독립검산,통과,독립 수식 기반 가치 계산 모두 일치
5,평가기준일,통과,"최근 확인 가능한 종가, 7일 이내"
6,공시선행성,통과,채택 공시 접수일이 주가 날짜를 넘지 않음
7,동종기간동일,통과,동종기업과 실적기간 동일
8,선택 상위분석,통과,종목코드 및 선택 제공 기업명·공시 전제 일치(미제공 허용)
9,공통표 교차대조,통과,독립 실행: 공통표 미입력으로 DART 원천 사용


## 10. 최종 HTML 리포트·감사 인계 패키지

HTML 보고서는 **현재 기준 마지막 확인 종가 날짜, 실적 기간, 공시 접수일**을 명시합니다. CSV와 JSON에는 공시 기준 날짜 및 상위 산업·기업 분석 연계 정보, 검증 경고를 기록합니다. 핵심 검증에서 실패하면 HTML을 만들지 않습니다.


In [16]:
OUT=f"{STOCK_CODE}_밸류에이션_v8_독립_통합인계"
os.makedirs(OUT,exist_ok=True)
def w(name,s):Path(OUT,name).write_text(str(s),encoding="utf-8")
# 차단되어도 실패원인은 인계 폴더에 남긴다
pd.DataFrame(LOG).to_csv(Path(OUT,"5_검증로그.csv"),index=False,encoding="utf-8-sig")
if not PROCEED:
    w("6_미확인_보류항목.md","분석/검증 실패: 보고서 생성 차단\n"+"\n".join(HARD_FAILURES+WARNINGS))
    print("⛔ 정식 리포트 생성 차단. 검증 로그 확인:",OUT)
else:
    limits=[]
    limits+=PRE.get("issues",[])+step1.get("data_issues",[])+step2.get("limitation",[])+step2.get("data_issues",[])
    limits += [f"현재가 비교: {VALUATION_PRICE[1]} 종가 / 마지막 확인 가능 가격. 최신 실적: {core[YEAR].get('valuation_period')} 종료 {core[YEAR].get('valuation_period_end')}, 공시 접수 {core[YEAR].get('valuation_filing_dt')}",
               "과거 PER은 연말가격과 사후 확정 연간 실적 조합(당시 이용 가능 투자 지표가 아님)",
               "DCF는 ΔNWC=0 고정, 5년 일정 성장률·마진·재투자율, 영구성장 가정",
               "감가상각비/CAPEX/이자부채는 자동 합산하지 않고 검토자 승인치만 사용",
               "동종기업 미제공 시 과거 자기기업 배수 3개 이상 중앙값으로 조건부 대체. 이는 동종기업 비교와 다르며 사후 연차확정치 사용",
               "최신 채택 공시의 보통주 유통주식수 사용(공시 이후 증자·소각·분할·가격 수정 여부 확인 필요; EPS는 가중평균 EPS와 차이 가능)",
               "사업부문 실적은 수동 입력, 보통주만 계산, 12월 결산 비금융업 전제"]
    limits+=WARNINGS
    if len(PEERS)<3:limits.append("비교기업 표본 3개 미만. 비교기업이 없으면 자기 회사 과거 배수 조건부 적용 여부를 확인하세요")
    if not UPSTREAM_ANALYSIS_BRIEF:limits.append("상위 산업·기업분석 미제공: 재무정보 기반 제한적 밸류에이션이며 산업별 사업가정은 교차 검토 필요")
    report_path=Path(OUT,"1_밸류에이션_보고서_v8.html")
    build_html_report(core[YEAR]["종목명"],VALUATION_PRICE[0],MODEL_TABLE,ACTUALS,FORECAST,
                      CALC[["모형","시나리오","주당가치(원)","괴리율(%)","극단값검사"]],
                      HIST5,PEERS,limits,AI_SUMMARY,CHART,report_path)
    dart_tagged.to_csv(Path(OUT,"2_원본_재무데이터_rid.csv"),index=False,encoding="utf-8-sig")
    dart_sources.to_csv(Path(OUT,"2_출처목록.csv"),index=False,encoding="utf-8-sig")
    metrics.to_csv(Path(OUT,"2_지표.csv"),index=False,encoding="utf-8-sig")
    MODEL_TABLE.to_csv(Path(OUT,"3_모형판정.csv"),index=False,encoding="utf-8-sig")
    w("3_추정_가정.json",json.dumps({"모형선정":step1,"가정":step2,"사업부매출":SEGMENTS,
                                     "수동계정검증":VERIFIED_FINANCIAL_OVERRIDES,"TTM수동확정":VERIFIED_TTM_OVERRIDES,"가격기준":VALUATION_PRICE,"평가실적기간":core[YEAR].get('valuation_period'),
                                     "평가실적종료":core[YEAR].get('valuation_period_end'),"평가공시접수일":core[YEAR].get('valuation_filing_dt'),
                                     "공통표":{"입력파일":COMMON_CSV_PATH or "미제공", "원본행수":int(len(COMMON_TABLE)) if COMMON_TABLE is not None else 0, "재무교차검증":{str(y):annual_core[y].get("common_trace",{}) for y in (YEAR-1,YEAR)}},
                                     "배수참고출처":MULTIPLE_REFERENCE_SOURCE,
                                     "상위분석":{"종목코드":STOCK_CODE,"기업명":UPSTREAM_COMPANY_NAME,"기업유형":UPSTREAM_COMPANY_TYPE,
                                                 "산업":UPSTREAM_INDUSTRY,"작성기준일":UPSTREAM_ANALYSIS_ASOF,
                                                 "인계근거":UPSTREAM_SOURCE_NOTE,"핵심분석":UPSTREAM_ANALYSIS_BRIEF,"선정동종기업":PEER_CODES},
                                     "동종배수허용구간":BANDS},ensure_ascii=False,indent=2,default=str))
    pd.DataFrame(SOURCE_TIMING+PEER_SOURCE_TIMING).to_csv(Path(OUT,'2_현재기준_가격공시시점.csv'),index=False,encoding='utf-8-sig')
    pd.DataFrame([{'계정':k,**v} for k,v in core[YEAR].get('ttm_lineage',{}).items()]).to_csv(
        Path(OUT,'2_TTM_원천계산내역.csv'),index=False,encoding='utf-8-sig')
    CALC.to_csv(Path(OUT,"4_시나리오_가치계산.csv"),index=False,encoding="utf-8-sig")
    FORECAST.to_csv(Path(OUT,"4_매출_이익_추정.csv"),index=False,encoding="utf-8-sig")
    SENSITIVITY.to_csv(Path(OUT,"4_DCF_WACC_g_민감도.csv"),index=False,encoding="utf-8-sig")
    w("6_미확인_보류항목.md","\n".join("- "+str(x) for x in limits if x))
    w("0_프롬프트_원문.txt","\n\n=====\n\n".join([prompt1,prompt2,prompt3]))
    print("✅ 검증 통과·저장:",sorted(os.listdir(OUT)))
    display(HTML(f'<a href="{report_path}" target="_blank">HTML 리포트 열기</a>'))


✅ 검증 통과·저장: ['0_프롬프트_원문.txt', '1_밸류에이션_보고서_v8.html', '2_TTM_원천계산내역.csv', '2_원본_재무데이터_rid.csv', '2_지표.csv', '2_출처목록.csv', '2_현재기준_가격공시시점.csv', '3_모형판정.csv', '3_추정_가정.json', '4_DCF_WACC_g_민감도.csv', '4_매출_이익_추정.csv', '4_시나리오_가치계산.csv', '5_검증로그.csv', '6_미확인_보류항목.md']


## 11. A4 한 장 디자인 요약 보고서 자동 생성

**별도 업로드 없이** 직전 단계의 계산값으로 HTML·A4 PDF를 자동 생성합니다. DART/Gemini는 다시 호출하지 않습니다. 자세한 근거와 검증 로그는 기존 v8 인계 폴더에 보존합니다.

In [17]:
#@title 🖨️ A4 한 장 요약 PDF 자동 생성 (업로드 불필요) { display-mode: "form" }
# -*- coding: utf-8 -*-
"""v8 최종 보고서 저장 직후 실행: A4 한 장 디자인 요약 PDF+HTML 자동 생성.
DART/Gemini API 추가 호출 없음. 재실행 시에도 업로드 필요 없음.
"""
from pathlib import Path
import html as _html, math as _math, base64 as _b64, subprocess as _subprocess, sys as _sys
import pandas as _pd

def _e(v):
    return _html.escape(str(v))

def _num(x, decimals=1, suffix=''):
    try:
        v=float(x)
        return f'{v:,.{decimals}f}{suffix}' if _math.isfinite(v) else '—'
    except (ValueError,TypeError): return '—'

def _maybe(df, col):
    return isinstance(df,_pd.DataFrame) and col in df.columns and not df.empty

def _cell(df, col, idx=0):
    try:return df.iloc[idx][col]
    except (AttributeError,IndexError,KeyError):return None

def _table(headers, rows, cls=''):
    h=''.join('<th>'+_e(x)+'</th>' for x in headers)
    b=''.join('<tr>'+''.join('<td>'+_e(v)+'</td>' for v in r)+'</tr>' for r in rows)
    return f'<table class="{cls}"><thead><tr>{h}</tr></thead><tbody>{b}</tbody></table>'

def _box(title, body, cls=''):
    return f'<section class="panel {cls}"><h2>{title}</h2><div class="inside">{body}</div></section>'

_missing=[x for x in ('PROCEED','CALC','MODEL_TABLE','VALUATION_PRICE','FORECAST','core','YEAR','STOCK_CODE') if x not in globals()]
if _missing:raise RuntimeError('v8 실행 결과 변수가 없습니다: '+', '.join(_missing))
if not PROCEED or not isinstance(CALC,_pd.DataFrame) or CALC.empty:
    raise RuntimeError('v8 검증/계산 성공 후 실행해 주세요. PROCEED와 CALC를 확인하세요.')

# Colab 환경에는 한글 폰트가 없을 수 있으므로 한 번만 설치
fonts=list(Path('/usr/share/fonts').rglob('*NanumGothic*'))
if not fonts:
    _subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'],check=True,stdout=_subprocess.DEVNULL)
try:
    from weasyprint import HTML as _HTML, CSS as _CSS
except ImportError:
    _subprocess.run([_sys.executable,'-m','pip','install','-q','weasyprint'],check=True)
    from weasyprint import HTML as _HTML, CSS as _CSS

c=core[YEAR]
company=str(c.get('종목명') or STOCK_CODE)
price,price_date=VALUATION_PRICE[:2]
period=str(c.get('valuation_period') or 'FY')
period_end=str(c.get('valuation_period_end') or '')
filing=str(c.get('valuation_filing_dt') or '')
primary=str(step1.get('primary_model','—')) if isinstance(globals().get('step1'),dict) else '—'
secondary=str(step1.get('secondary_model','—')) if isinstance(globals().get('step1'),dict) else '—'
status_rows=[]
for _,r in MODEL_TABLE.iterrows():
    status_rows.append([r.get('model','—'),r.get('final_status','—'),str(r.get('code_reason','—'))[:65]])
model_block=_table(['모형','판정','코드 판정 근거'],status_rows)
model_block+=f'<div class="tiny mt">주력: <strong>{_e(primary)}</strong> · 보조: <strong>{_e(secondary)}</strong> · AI 판정은 코드 상한을 초과할 수 없음</div>'

cur=globals().get('CUR') or {}
hist=globals().get('HIST5')
peers=globals().get('PEERS')
multiples=['PER','PBR','EV/EBITDA','PSR']
mult_rows=[]
for m in multiples:
    val=cur.get(m) if isinstance(cur,dict) else None
    med=None
    if _maybe(hist,m):
        s=_pd.to_numeric(hist[m],errors='coerce').dropna();s=s[s>0]
        if not s.empty:med=s.median()
    pmed=None
    if _maybe(peers,m):
        s=_pd.to_numeric(peers[m],errors='coerce').dropna();s=s[s>0]
        if not s.empty:pmed=s.median()
    mult_rows.append([m,_num(val,2),_num(med,2),_num(pmed,2)])
mult_block=_table(['배수','현재','과거 5년 중앙값','동종 중앙값'],mult_rows)
mult_block+='<div class="tiny mt">과거 배수는 사후 확정 연간실적 기준이며 당시 실시간 이용가능 지표가 아님</div>'

chart_html=''
chart_path=globals().get('CHART')
if chart_path and Path(str(chart_path)).exists():
    content=Path(str(chart_path)).read_bytes()
    chart_html='<img class="chart" src="data:image/png;base64,'+_b64.b64encode(content).decode('ascii')+'">'
else:
    chart_html='<div class="empty">5개년 추이 차트 미확보</div>'

if isinstance(peers,_pd.DataFrame) and not peers.empty:
    prows=[]
    for _,r in peers.head(5).iterrows():
        prows.append([str(r.get('종목명',r.get('종목코드','—')))[:12], _num(r.get('PER'),2),_num(r.get('PBR'),2),_num(r.get('EV/EBITDA'),2)])
    peer_block=_table(['비교기업','PER','PBR','EV/EBITDA'],prows)
else:
    peer_block='<div class="empty">동종기업 미제공 · 과거 자기배수는 조건부 참고치</div>'

forecast=globals().get('FORECAST')
frows=[]
for scen in ('bear','base','bull'):
    if isinstance(forecast,_pd.DataFrame) and '시나리오' in forecast.columns:
        part=forecast[forecast['시나리오']==scen]
        if not part.empty:
            r=part.iloc[0]
            frows.append([scen.title(),_num(r.get('예상매출(억원)'),0),_num(r.get('예상영업이익(억원)'),0),_num(r.get('예상EPS(원)'),0),_num(r.get('영업이익률(%)'),1,'%')])
forecast_block=_table(['구분','매출(억원)','영업익(억원)','EPS(원)','영업이익률'],frows)

if _maybe(CALC,'주당가치(원)'):
    calc_rows=[]
    for m in [primary,secondary]:
        if m=='—':continue
        for scen in ('bear','base','bull'):
            sel=CALC[(CALC['모형']==m)&(CALC['시나리오']==scen)]
            if sel.empty:continue
            r=sel.iloc[0]
            calc_rows.append([m,scen.title(),_num(r.get('주당가치(원)'),0),_num(r.get('괴리율(%)'),1,'%')])
else: calc_rows=[]
if not calc_rows:
    for _,r in CALC.head(9).iterrows():calc_rows.append([r.get('모형'),r.get('시나리오'),_num(r.get('주당가치(원)'),0),_num(r.get('괴리율(%)'),1,'%')])
value_block=_table(['모형','시나리오','주당가치(원)','괴리율'],calc_rows)

base_value=CALC[(CALC['모형']==primary)&(CALC['시나리오']=='base')] if primary!='—' else _pd.DataFrame()
if base_value.empty:base_value=CALC[CALC['시나리오']=='base']
base_val=base_value.iloc[0]['주당가치(원)'] if not base_value.empty else None
base_gap=base_value.iloc[0]['괴리율(%)'] if not base_value.empty else None
base_str=_num(base_val,0,'원')

type_ = globals().get('UPSTREAM_COMPANY_TYPE') or '미제공'
industry = globals().get('UPSTREAM_INDUSTRY') or '미제공'
summary=globals().get('AI_SUMMARY','')
summary=' '.join(str(summary).replace('**','').replace('\n',' ').split())
if summary.startswith('AI 해석 생성 실패'):summary='AI 해석 미완료 (계산 결과만 표시)'
summary=_e(summary[:430]) if summary else 'AI 해석 내용 미제공'
warn=globals().get('WARNINGS',[]) or []
warning_text=' · '.join(str(w) for w in warn[:3]) if warn else '별도 경고 없음'

outdir=Path(str(globals().get('OUT',f'{STOCK_CODE}_밸류에이션_v8_독립_통합인계')))
outdir.mkdir(parents=True,exist_ok=True)
report_html=outdir/'1_밸류에이션_디자인요약_A4.html'
report_pdf=outdir/'1_밸류에이션_디자인요약_A4.pdf'

html=f'''<!doctype html><html lang="ko"><head><meta charset="utf-8"><title>{_e(company)} 밸류에이션 요약</title></head>
<body><header class="top"><b>KFA AI 리서치센터</b><span>기업 리서치 · 밸류에이션 요약</span><em>04 / VALUATION</em></header>
<div class="page">
<div class="head"><div><h1>{_e(company)} <small>({_e(STOCK_CODE)})</small> <i>|</i> 기업가치 평가 리포트</h1><p>최신 확인 종가와 공개된 {_e(period)} 재무정보를 활용한 조건부 가치평가</p></div><div class="price"><div><span>평가 종가일</span><b>{_e(price_date)}</b></div><div><span>현재 종가</span><b>{_num(price,0)}원</b></div><div><span>실적 기준</span><b>{_e(period)} / {_e(period_end)}</b></div></div></div>
{_box('① 기업 특성 및 평가모형 선정',f'<div class="twocol"><div>'+_table(['항목','정보'],[['기업 유형',type_],['산업',industry],['현재 실적기간',period],['접수일',filing]])+'</div><div>'+model_block+'</div></div>','wide')}
<div class="two">{_box('② 현재 배수와 과거 비교',mult_block)}{_box('③ 평가배수 추이 · PER (5년)',chart_html)}</div>
<div class="two">{_box('④ 동종기업 비교',peer_block)}{_box('⑤ 1년 선행 이익 및 가정',forecast_block)}</div>
<div class="two value">{_box('⑥ 시나리오별 가치평가',value_block)}{_box('⑦ 평가 기준 및 주요 주의사항',f'<div class="notice"><b>주력 모형 { _e(primary) }</b><p>Base 주당가치 <strong>{base_str}</strong></p><p>최근 확인 종가 대비 괴리율 <strong>{_num(base_gap,1,"%")}</strong></p><p>비교기업 부재, DCF 필수 계정 미확정 시 해당 평가모형은 보류됩니다.</p></div>')}</div>
{_box('⑧ 종합 해석 및 검증 포인트',f'<div class="summary"><div><b>요약 의견 (자동 생성)</b><p>{summary}</p></div><div><b>추가 확인</b><p>{_e(warning_text)}</p><p>코드 검증 통과 ≠ 추정 가정의 경제적 타당성 보장</p></div></div>','wide')}
<footer><div><b>데이터</b> OpenDART 재무공시 · 확인 가능한 시장 종가 · v8 계산 로그</div><div><b>범위</b> 연구·학습 목적, 투자 권유 아님 · 상세 근거는 원본 CSV/검증로그 참조</div></footer>
</div></body></html>'''

css=r'''
@font-face{font-family:Nanum;src:local('NanumGothic')}
@page{size:A4 portrait;margin:0}
*{box-sizing:border-box}
html,body{margin:0;padding:0;color:#182541;font-family:Nanum,'Nanum Gothic',sans-serif}
body{font-size:8px;background:#fff}
.top{height:36px;display:flex;align-items:center;gap:13px;padding:0 20px;color:white;background:#1c2c4d;font-size:10px}
.top span{font-size:9px}.top em{margin-left:auto;font-size:10px;color:#e0ceff;font-style:normal;font-weight:bold}
.page{padding:11px 12px 9px}
.head{display:flex;justify-content:space-between;align-items:flex-start;gap:15px;margin:0 3px 11px}
.head h1{font-size:18px;margin:4px 0 5px;letter-spacing:-.6px;white-space:nowrap}
.head h1 small{font-size:14px}.head h1 i{font-style:normal;color:#a9acc7;font-weight:normal}
.head p{margin:0;font-size:8.7px;color:#53627c}
.price{width:178px;flex-shrink:0;border:1px solid #c7c7e2;border-radius:8px;padding:5px 10px;background:#f2f3fc}
.price>div{display:flex;justify-content:space-between;gap:6px;padding:2px 0;font-size:8.1px}
.price span{color:#4c5877}.price b{color:#152543}
.panel{background:white;border:1px solid #d5d7eb;border-radius:9px;overflow:hidden;min-width:0;margin-bottom:9px}
.panel h2{margin:0;padding:7px 10px;color:white;background:#353368;font-size:11px;line-height:1.1}
.inside{padding:7px 9px}
.two{display:grid;grid-template-columns:1fr 1fr;gap:10px;align-items:start}
.value{grid-template-columns:1.22fr .78fr}
.twocol{display:grid;grid-template-columns:.78fr 1.22fr;gap:10px}
table{border-collapse:collapse;width:100%;font-size:7.4px;table-layout:auto}
th,td{border-bottom:1px solid #e2e4ee;padding:4px 3px;text-align:left;vertical-align:middle;overflow-wrap:anywhere}
th{background:#eff0fa;color:#202846;font-weight:bold}
tr:nth-child(even) td{background:#fafaff}
.chart{width:100%;height:139px;object-fit:contain;display:block}
.empty{background:#f5f5fa;border-radius:5px;padding:15px 8px;color:#596377}
.tiny{font-size:7.1px;color:#667189;line-height:1.35}
.mt{margin-top:5px}
.notice{padding:4px 5px;background:#f2f0ff;border-radius:7px;line-height:1.45}
.notice b{color:#3b258c}.notice strong{color:#38268d;font-size:11px}
.notice p{margin:5px 0}
.summary{display:grid;grid-template-columns:1.42fr 1fr;gap:12px;line-height:1.38}
.summary>div+div{border-left:1px solid #d9d9e7;padding-left:11px}
.summary b{color:#36276a;font-size:9px}.summary p{margin:4px 0;font-size:7.7px}
footer{display:flex;gap:11px;justify-content:space-between;font-size:7.1px;line-height:1.35;color:#4a566e;border-top:1px solid #e3e2ee;padding-top:5px}
footer>div{width:50%}
'''
report_html.write_text('<style>'+css+'</style>'+html,encoding='utf-8')
# 레이아웃만 점진적으로 축소한다. 실제 출력이 1쪽인지 검사한다.
last_pages=None
for zoom in (1.0,.97,.94,.91,.88,.85,.82,.79,.76,.73):
    zoomcss=f'body{{zoom:{zoom};}}'
    doc=_HTML(string=html,base_url=str(outdir.resolve())).render(stylesheets=[_CSS(string=css+zoomcss)])
    last_pages=len(doc.pages)
    if last_pages==1:
        doc.write_pdf(str(report_pdf))
        print(f'✅ A4 세로 1장 요약 PDF: {report_pdf} (배율 {zoom:.0%})')
        print(f'✅ 미리보기 HTML: {report_html}')
        break
else:
    print(f'⚠️ 내용이 {last_pages}페이지입니다. 숫자/내용 생략 없이 한 장 배치 실패; HTML은 생성됨.')

# Colab이면 다운로드
if report_pdf.exists():
    try:
        from google.colab import files
        files.download(str(report_pdf))
    except ImportError:
        pass


✅ A4 세로 1장 요약 PDF: 004370_밸류에이션_v8_독립_통합인계/1_밸류에이션_디자인요약_A4.pdf (배율 100%)
✅ 미리보기 HTML: 004370_밸류에이션_v8_독립_통합인계/1_밸류에이션_디자인요약_A4.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## v8 사용 범위 및 최종 통합 안내

- **테스트 셀에서 종목코드만으로 독립 실행 가능**: DART와 주가 데이터를 수집해 평가. 상위 산업·기업분석·비교기업의 유무는 그 자체로 차단 사유가 아닙니다.
- **공통 15열 표 연계**: `VALUATION_HANDOFF["common_records"]`에 지정한 표의 종목코드, 연결 기준, 공시일, 계정, 단위, 상태를 검사합니다. 매출액·영업이익·지배주주순이익·지배자본·현금은 공통표에서 검증 가능할 때 값과 출처를 채택하며, 실제 DART 표준 계정과 충돌하면 중단합니다.
- **공통표만으로 모든 모형 계산은 불가능**: 일반적인 공통 15열 표에 주식수·CAPEX·D&A·순차입금·분기별 누적 실적이 없는 경우 DART 조회로 보완합니다. 원천 확인 불가 모형은 보류합니다.
- **동종기업이 없을 때**: 자기 회사의 역사적 PER/PBR/PSR 등 유효 연도 3개 이상을 조건부 배수 기준으로 사용합니다. 이는 동종기업 기반 가치평가를 대체한 추정이며 경제적 타당성 검증은 별도입니다.
- **현재 시점**: 최근 확인 가능한 종가와 해당일 전에 공개된 최근 12개월(TTM) 또는 FY 실적을 사용합니다. 구체적인 가격일과 채택 공시일을 보고서 및 인계 파일에 기록합니다.
- **고정 제한**: 12월 결산 비금융 한국 상장회사, 보통주 기준. DCF ΔNWC=0, 자동 상세계정 합산 미지원. Gemini가 만든 시나리오는 실제 애널리스트 실적 전망치가 아니며 근거의 경제적 타당성은 별도 검증 대상입니다.
- **검증**: 코드·수식 단위 테스트와 실제 DART·Gemini 통합 실행은 별개입니다. 실제 API 키로 Colab에서 한 기업의 보고서·로그를 최종 확인한 다음 제출하세요.


### v8 Gemini 자동 재시도
일시적 HTTP 503/502/504/500 및 네트워크 타임아웃과 **일시적인** HTTP 429만 자동 재시도합니다. 기본/예비 모델을 번갈아 호출하고 대기 시간을 5→10→20→40→80→120초(최대)로 늘립니다. 인증·요청 형식·존재하지 않는 모델·영구 할당량 제한은 자동 해결할 수 없으므로 중단합니다. Colab 런타임 연결이 끊기거나 사용자가 ■ 중지 버튼을 누르면 재시도도 중단됩니다. 응답을 받았다고 해도 가정과 계산 검증은 기존 규칙대로 수행합니다.
